In [1]:
# ── Force-kill uvloop before anything else imports it ──
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "uvloop", "-y"], capture_output=True)

# Prevent reimport if it's cached
import sys as _sys
_sys.modules['uvloop'] = None

import asyncio
asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

In [2]:
# ── Install nanoathens SDK + all dependencies ──
!pip install -q gradio==5.50.0
!pip uninstall uvloop -y
!pip install -q uvicorn==0.34.0
!pip install -q nanoathens==0.2.2 json-repair nest-asyncio scikit-image
!pip install -q transformers==5.0.0 accelerate huggingface_hub faiss-cpu pydantic openai scikit-learn matplotlib numpy gradio
!pip install -q reportlab 
print("✓ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 72.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.0 MB/s eta 0:00:0000:010:01
✓ All packages installed


### **Installations & Imports**

In [3]:
# (Installations handled in Cell 0)
print("✓ Ready")

✓ Ready


In [4]:
import os, json, re, time, asyncio, base64, uuid
import numpy as np
import faiss
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
from typing import Dict, List, Optional, Any, Callable
from pydantic import BaseModel
from enum import Enum

# ── nest_asyncio (required for Gradio in Kaggle/Colab) ──
import nest_asyncio
nest_asyncio.apply()

# ── nanoathens SDK (pip-installed) ──
from nanoathens import (
    ToolRegistry, ToolSchema, ToolType, ArgExtractorType,
    ContextBank, LLMValueExtractor,
    GroundedArgumentFiller,
    DataFlowEngine,
    GoalKeyResolver,
    DeclarativeDataFlowAgent, ToolRAGAgent,
    SessionStore, SESSION_STORE,
    run_medgemma, load_medgemma, set_pipeline,
)

print(f"✓ nanoathens v{__import__('nanoathens').__version__} imported")
print(f"  DeclarativeDataFlowAgent: {DeclarativeDataFlowAgent}")
print(f"  ToolRegistry: {ToolRegistry}")

✓ nanoathens v0.2.0 imported
  DeclarativeDataFlowAgent: <class 'nanoathens.agent.DeclarativeDataFlowAgent'>
  ToolRegistry: <class 'nanoathens.core.ToolRegistry'>


In [6]:
from huggingface_hub import login
#from google.colab import userdata
from kaggle_secrets import UserSecretsClient
# HF_TOKEN = userdata.get("HF_TOKEN")
secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
#login()
print("✓ HuggingFace login successful")



✓ HuggingFace login successful


### **Model Downloads & CheXpert Embedding Configuration**

In [7]:
import torch
from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModel
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# ── MedGemma (for analysis) ──
medgemma_pipe = hf_pipeline(
    "image-text-to-text",
    model="google/medgemma-1.5-4b-it",
    device=device,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
)
set_pipeline(medgemma_pipe)  # Register with nanoathens
print("✓ MedGemma loaded")

# ── MedSigLIP (for retrieval) ──
SIGLIP_MODEL = "google/medsiglip-448"
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
siglip_model     = AutoModel.from_pretrained(SIGLIP_MODEL, torch_dtype=torch.float16).to(device).eval()
print("✓ MedSigLIP loaded")

Device: cuda | GPU: Tesla P100-PCIE-16GB


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

✓ MedSigLIP loaded


In [9]:
# ── Paths ──
MEDSIG_LIP_SIZE  = (448, 448)
EMBED_BATCH_SIZE = 128
NUM_WORKERS      = 4
KAGGLE_ROOT      = "/kaggle/input/datasets/ashery/chexpert" 
LOCAL_INDEX_PATH  = "/kaggle/working/chexpert_train.index"
LOCAL_META_PATH   =  "/kaggle/working/chexpert_train_meta.npz"

# ── CheXpert label columns (the 14 competition labels) ──
LABEL_COLS = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

# ── Load CSVs ──
TRAIN_CSV = os.path.join(KAGGLE_ROOT, "train.csv")
VALID_CSV = os.path.join(KAGGLE_ROOT, "valid.csv")
train_df  = pd.read_csv(TRAIN_CSV)
valid_df  = pd.read_csv(VALID_CSV)

train_paths = [
    os.path.join(KAGGLE_ROOT, p.replace("CheXpert-v1.0-small/", ""))
    for p in train_df["Path"].tolist()
]
valid_paths = [
    os.path.join(KAGGLE_ROOT, p.replace("CheXpert-v1.0-small/", ""))
    for p in valid_df["Path"].tolist()
]

print(f"Train images: {len(train_paths)}")
print(f"Valid images: {len(valid_paths)}")
print(f"Label columns: {len(LABEL_COLS)}")

Train images: 223414
Valid images: 234
Label columns: 14


In [10]:
PRE_SAVED_INDEX = "/kaggle/input/datasets/boochijr/embeddings/chexpert_train.index"
PRE_SAVED_META = "/kaggle/input/datasets/boochijr/embeddings/chexpert_train_meta.npz"
index = faiss.read_index(PRE_SAVED_INDEX)
path_store = np.load(PRE_SAVED_META, allow_pickle=True)["path_store"].item()
print(f"Index size: {index.ntotal}")

Index size: 10000


In [11]:
index, path_store
def build_label_description(row: pd.Series) -> str:
    present, absent, uncertain = [], [], []
    for col in LABEL_COLS:
        val = row.get(col, np.nan)
        if pd.isna(val): continue
        val = float(val)
        if val == 1.0:    present.append(col)
        elif val == 0.0:  absent.append(col)
        elif val == -1.0: uncertain.append(col)

    parts = []
    if present:   parts.append(f"Present: {', '.join(present)}.")
    if uncertain: parts.append(f"Uncertain: {', '.join(uncertain)}.")
    if absent:    parts.append(f"Absent: {', '.join(absent)}.")
    if not parts: parts.append("No findings documented.")

    meta = []
    for field in ["Sex", "Age", "Frontal/Lateral", "AP/PA"]:
        if field in row and not pd.isna(row.get(field)):
            meta.append(f"{field}: {row[field]}")
    if meta:
        parts.insert(0, " | ".join(meta) + ".")
    return " ".join(parts)


def build_description_lookup(csv_paths):
    lookup = {}
    for csv_path in csv_paths:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            key = row["Path"].replace("CheXpert-v1.0-small/", "")
            lookup[key] = build_label_description(row)
    return lookup


def embed_query_image(image_path: str) -> np.ndarray:
    img    = Image.open(image_path).convert("RGB").resize(MEDSIG_LIP_SIZE, Image.BICUBIC)
    inputs = siglip_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = siglip_model.get_image_features(**inputs)
        emb = outputs if torch.is_tensor(outputs) else getattr(outputs, "pooler_output", outputs[0])
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().float().numpy().astype("float32")


def faiss_retrieve(query_vec, k=3):
    scores, indices = index.search(query_vec.reshape(1, -1), k)
    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
        if idx == -1: continue
        rel_path = path_store[idx]
        results.append({
            "rank": rank, "score": round(float(score), 4),
            "path": rel_path,
            "description": description_lookup.get(rel_path, "No description."),
        })
    return results


# Build lookup
description_lookup = build_description_lookup([TRAIN_CSV, VALID_CSV])
print(f"✓ Description lookup built | {len(description_lookup)} entries")

✓ Description lookup built | 223648 entries


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
#  JSON REPAIR UTILITY
#  Robust JSON extraction with recursive 
# ══════════════════════════════════════════════════════════════════════════════

def extract_and_validate_json(
    raw_text: str,
    pydantic_model: type = None,
    llm_caller: Callable = None,
    max_retries: int = 5,
) -> Dict:
    """Extract JSON from LLM output with recursive retry on failure.

    Strategy:
      1. Try direct JSON extraction from response
      2. If Pydantic validation fails, ask LLM to fix the JSON (up to max_retries)
      3. If fix fails twice on same error, request fresh inference

    Args:
        raw_text: Raw LLM response containing JSON
        pydantic_model: Optional Pydantic model class for validation
        llm_caller: LLM function for retry/fix (run_medgemma signature)
        max_retries: Maximum fix attempts

    Returns:
        Validated dict or best-effort parsed dict
    """
    def _try_parse(text: str) -> Optional[Dict]:
        """Try to extract JSON from text."""
        # Try json-repair if available
        try:
            from json_repair import repair_json
            repaired = repair_json(text, return_objects=True)
            if isinstance(repaired, dict):
                return repaired
        except ImportError:
            pass

        # Manual extraction
        # Look for ```json ... ``` blocks first
        json_match = re.search(r"```json\s*(.*?)```", text, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(1).strip())
            except json.JSONDecodeError:
                pass

        # Look for { ... } blocks
        s = text.find("{")
        e = text.rfind("}")
        if s != -1 and e > s:
            try:
                return json.loads(text[s : e + 1])
            except json.JSONDecodeError:
                pass

        return None

    # Attempt 1: Direct parse
    parsed = _try_parse(raw_text)
    if parsed is None:
        parsed = {}

    # If no Pydantic model, return as-is
    if pydantic_model is None:
        return parsed

    # Validate with Pydantic
    try:
        validated = pydantic_model(**parsed)
        return validated.model_dump()
    except Exception as validation_error:
        last_error = str(validation_error)

    # Retry loop: ask LLM to fix
    if llm_caller is None:
        return parsed  # Can't fix without LLM

    for attempt in range(max_retries):
        fix_prompt = (
            f"The following JSON failed validation.\n"
            f"ERROR: {last_error}\n"
            f"EXPECTED SCHEMA: {pydantic_model.model_json_schema()}\n"
            f"BROKEN JSON: {json.dumps(parsed) if parsed else raw_text[:500]}\n\n"
            f"Return ONLY the corrected JSON. No explanation."
        )
        messages = [
            {"role": "user", "content": [{"type": "text", "text": fix_prompt}]}
        ]
        try:
            fix_response = llm_caller(
                messages=messages, max_new_tokens=512, temperature=0.0
            )
            fixed = _try_parse(fix_response)
            if fixed:
                validated = pydantic_model(**fixed)
                return validated.model_dump()
        except Exception as e:
            new_error = str(e)
            if new_error == last_error:
                # Same error twice — need fresh inference, not a fix
                break
            last_error = new_error

    # Return best effort
    return parsed


print("JSON repair utility defined")

JSON repair utility defined


### **Co-Radiologist Agent: Image Review Skills**

In [13]:
# ─────────── Chest Xray Review Skills ─────────────────
class Ternary(str, Enum):
    yes = "yes"; no = "no"; unclear = "unclear"

class CXRReview(BaseModel):
    no_finding: Ternary
    enlarged_cardiomediastinum: Ternary
    cardiomegaly: Ternary
    lung_opacity: Ternary
    lung_lesion: Ternary
    edema: Ternary
    consolidation: Ternary
    pneumonia: Ternary
    atelectasis: Ternary
    pneumothorax: Ternary
    pleural_effusion: Ternary
    pleural_other: Ternary
    fracture: Ternary
    support_devices: Ternary
    verification_confidence: str = "medium"
    verification_notes: str = ""

# Map CXRReview fields → LABEL_COLS index (exact 14-field correspondence)
CXRREVIEW_TO_LABEL = {
    "no_finding": "No Finding",
    "enlarged_cardiomediastinum": "Enlarged Cardiomediastinum",
    "cardiomegaly": "Cardiomegaly",
    "lung_opacity": "Lung Opacity",
    "lung_lesion": "Lung Lesion",
    "edema": "Edema",
    "consolidation": "Consolidation",
    "pneumonia": "Pneumonia",
    "atelectasis": "Atelectasis",
    "pneumothorax": "Pneumothorax",
    "pleural_effusion": "Pleural Effusion",
    "pleural_other": "Pleural Other",
    "fracture": "Fracture",
    "support_devices": "Support Devices",
}


def cxr_review_to_binary(review_dict: dict) -> list:
    """Convert CXRReview dict to binary label vector aligned with LABEL_COLS."""
    result = []
    for field, label in CXRREVIEW_TO_LABEL.items():
        val = review_dict.get(field, "no")
        result.append(1 if str(val).lower() == "yes" else 0)
    return result


CXR_ANALYSIS_PROMPT = """You are an expert radiologist. Analyze this chest X-ray.
For each finding, respond with exactly "yes", "no", or "unclear".

Return ONLY a JSON object with these exact fields:
{
    "no_finding": "yes/no/unclear",
    "enlarged_cardiomediastinum": "yes/no/unclear",
    "cardiomegaly": "yes/no/unclear",
    "lung_opacity": "yes/no/unclear",
    "lung_lesion": "yes/no/unclear",
    "edema": "yes/no/unclear",
    "consolidation": "yes/no/unclear",
    "pneumonia": "yes/no/unclear",
    "atelectasis": "yes/no/unclear",
    "pneumothorax": "yes/no/unclear",
    "pleural_effusion": "yes/no/unclear",
    "pleural_other": "yes/no/unclear",
    "fracture": "yes/no/unclear",
    "support_devices": "yes/no/unclear",
    "verification_confidence": "low/medium/high",
    "verification_notes": "brief explanation"
}"""


print(f"✓ CXRReview schema: {len(CXRREVIEW_TO_LABEL)} labels mapped to LABEL_COLS")

✓ CXRReview schema: 14 labels mapped to LABEL_COLS


In [14]:
# ─────────── CT Head Review Skills ─────────────────

class CTHeadReview(BaseModel):
    hemorrhage: Ternary
    intraventricular_hemorrhage: Ternary
    subarachnoid_hemorrhage: Ternary
    subdural_hemorrhage: Ternary
    epidural_hemorrhage: Ternary
    calvarial_fracture: Ternary
    midline_shift: Ternary
    mass_effect: Ternary
    hydrocephalus: Ternary
    ischemic_stroke: Ternary
    cerebral_edema: Ternary
    intracranial_mass: Ternary
    verification_confidence: str = "medium"
    verification_notes: str = ""

CTHEAD_TO_LABEL = {
    "hemorrhage": "Hemorrhage",
    "intraventricular_hemorrhage": "Intraventricular Hemorrhage",
    "subarachnoid_hemorrhage": "Subarachnoid Hemorrhage",
    "subdural_hemorrhage": "Subdural Hemorrhage",
    "epidural_hemorrhage": "Epidural Hemorrhage",
    "calvarial_fracture": "Calvarial Fracture",
    "midline_shift": "Midline Shift",
    "mass_effect": "Mass Effect",
    "hydrocephalus": "Hydrocephalus",
    "ischemic_stroke": "Ischemic Stroke",
    "cerebral_edema": "Cerebral Edema",
    "intracranial_mass": "Intracranial Mass",
}

def ct_head_review_to_binary(review_dict: dict) -> list:
    """Convert CTHeadReview dict to binary label vector aligned with CTHEAD_LABEL_COLS."""
    result = []
    for field in CTHEAD_TO_LABEL:
        val = review_dict.get(field, "no")
        result.append(1 if str(val).lower() == "yes" else 0)
    return result

CT_HEAD_ANALYSIS_PROMPT = """You are an expert neuroradiologist. Analyze this CT head scan.
For each finding, respond with exactly "yes", "no", or "unclear".
Return ONLY a JSON object with these exact fields:
{
    "hemorrhage": "yes/no/unclear",
    "intraventricular_hemorrhage": "yes/no/unclear",
    "subarachnoid_hemorrhage": "yes/no/unclear",
    "subdural_hemorrhage": "yes/no/unclear",
    "epidural_hemorrhage": "yes/no/unclear",
    "calvarial_fracture": "yes/no/unclear",
    "midline_shift": "yes/no/unclear",
    "mass_effect": "yes/no/unclear",
    "hydrocephalus": "yes/no/unclear",
    "ischemic_stroke": "yes/no/unclear",
    "cerebral_edema": "yes/no/unclear",
    "intracranial_mass": "yes/no/unclear",
    "verification_confidence": "low/medium/high",
    "verification_notes": "brief explanation"
}"""

CTHEAD_LABEL_COLS = list(CTHEAD_TO_LABEL.values())
print(f"✓ CTHeadReview schema: {len(CTHEAD_TO_LABEL)} labels")


✓ CTHeadReview schema: 12 labels


In [15]:
# ─────────── CT Abdominal Review Skills ─────────────────

class CTAbdominalReview(BaseModel):
    bowel_obstruction: Ternary
    appendicitis: Ternary
    diverticulitis: Ternary
    hepatic_mass: Ternary
    hepatic_steatosis: Ternary
    splenomegaly: Ternary
    hydronephrosis: Ternary
    renal_calculi: Ternary
    cholecystitis: Ternary
    gallstones: Ternary
    ascites: Ternary
    pneumoperitoneum: Ternary
    aortic_aneurysm: Ternary
    active_extravasation: Ternary
    lymphadenopathy: Ternary
    verification_confidence: str = "medium"
    verification_notes: str = ""


CTABDO_TO_LABEL = {
    "bowel_obstruction": "Bowel Obstruction",
    "appendicitis": "Appendicitis",
    "diverticulitis": "Diverticulitis",
    "hepatic_mass": "Hepatic Mass",
    "hepatic_steatosis": "Hepatic Steatosis",
    "splenomegaly": "Splenomegaly",
    "hydronephrosis": "Hydronephrosis",
    "renal_calculi": "Renal Calculi",
    "cholecystitis": "Cholecystitis",
    "gallstones": "Gallstones",
    "ascites": "Ascites",
    "pneumoperitoneum": "Pneumoperitoneum",
    "aortic_aneurysm": "Aortic Aneurysm",
    "active_extravasation": "Active Extravasation",
    "lymphadenopathy": "Lymphadenopathy",
}

def ct_abdo_review_to_binary(review_dict: dict) -> list:
    """Convert CTAbdominalReview dict to binary label vector aligned with CTABDO_LABEL_COLS."""
    result = []
    for field in CTABDO_TO_LABEL:
        val = review_dict.get(field, "no")
        result.append(1 if str(val).lower() == "yes" else 0)
    return result


CT_ABDOMINAL_ANALYSIS_PROMPT = """You are an expert abdominal radiologist. Analyze this CT abdomen/pelvis.
For each finding, respond with exactly "yes", "no", or "unclear".
Return ONLY a JSON object with these exact fields:
{
    "bowel_obstruction": "yes/no/unclear",
    "appendicitis": "yes/no/unclear",
    "diverticulitis": "yes/no/unclear",
    "hepatic_mass": "yes/no/unclear",
    "hepatic_steatosis": "yes/no/unclear",
    "splenomegaly": "yes/no/unclear",
    "hydronephrosis": "yes/no/unclear",
    "renal_calculi": "yes/no/unclear",
    "cholecystitis": "yes/no/unclear",
    "gallstones": "yes/no/unclear",
    "ascites": "yes/no/unclear",
    "pneumoperitoneum": "yes/no/unclear",
    "aortic_aneurysm": "yes/no/unclear",
    "active_extravasation": "yes/no/unclear",
    "lymphadenopathy": "yes/no/unclear",
    "verification_confidence": "low/medium/high",
    "verification_notes": "brief explanation"
}"""

CTABDO_LABEL_COLS = list(CTABDO_TO_LABEL.values())
print(f"✓ CTAbdominalReview schema: {len(CTABDO_TO_LABEL)} labels")

✓ CTAbdominalReview schema: 15 labels


In [16]:
# ─────────── Unified Prompt Skill Selector ─────────────────

ANALYSIS_PROMPT_MAP = {
    ("xray", "chest"): CXR_ANALYSIS_PROMPT,
    ("ct", "head"): CT_HEAD_ANALYSIS_PROMPT,
    ("ct", "brain"): CT_HEAD_ANALYSIS_PROMPT,
    ("ct", "abdomen"): CT_ABDOMINAL_ANALYSIS_PROMPT,
}

REVIEW_SCHEMA_MAP = {
    ("xray", "chest"): CXRReview,
    ("ct", "head"): CTHeadReview,
    ("ct", "brain"): CTHeadReview,
    ("ct", "abdomen"): CTAbdominalReview,
}

LABEL_MAP = {
    ("xray", "chest"): CXRREVIEW_TO_LABEL,
    ("ct", "head"): CTHEAD_TO_LABEL,
    ("ct", "brain"): CTHEAD_TO_LABEL,
    ("ct", "abdomen"): CTABDO_TO_LABEL,
}

BINARY_CONVERTER_MAP = {
    ("xray", "chest"): cxr_review_to_binary,
    ("ct", "head"): ct_head_review_to_binary,
    ("ct", "brain"): ct_head_review_to_binary,
    ("ct", "abdomen"): ct_abdo_review_to_binary,
}

In [16]:
def get_analysis_prompt(image_type: str, body_part: str = "chest") -> str:
    """Select the analysis prompt for a given modality and body part."""
    key = (image_type.lower().strip(), body_part.lower().strip())
    return ANALYSIS_PROMPT_MAP.get(key, CXR_ANALYSIS_PROMPT)

def get_review_schema(image_type: str, body_part: str = "chest"):
    """Select the Pydantic schema for a given modality and body part."""
    key = (image_type.lower().strip(), body_part.lower().strip())
    return REVIEW_SCHEMA_MAP.get(key, CXRReview)

def get_binary_converter(image_type: str, body_part: str = "chest"):
    """Select the binary converter for a given modality and body part."""
    key = (image_type.lower().strip(), body_part.lower().strip())
    return BINARY_CONVERTER_MAP.get(key, cxr_review_to_binary)

### **Co-Radiologist Agent: Tool Functions**

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
#  TOOL FUNCTIONS — Gemma Co-Radiologist
#  All 15 tools with modality-adaptive logic for CXR / CT Head / CT Abdomen
# ══════════════════════════════════════════════════════════════════════════════

# ────────────── Tool 1: retrieve similar images from Embeddings ────────────────
def _retrieve_similar_images(patient_image: str, image_type: str, k: int = 5) -> str:
    """Retrieve similar images using MedSigLIP + FAISS.

    NOTE: FAISS index currently contains CheXpert (CXR) data only.
    For non-CXR modalities, retrieval still runs but results will be
    CXR-based — downstream tools handle this gracefully.
    """
    if os.path.exists(patient_image):
        query_vec  = embed_query_image(patient_image)
        query_type = "image"
    else:
        # Text fallback — adapt to modality instead of hardcoding "chest radiograph"
        modality_text_map = {
            "xray": "chest radiograph",
            "ct": "CT scan medical image",
            "mri": "MRI scan medical image",
        }
        text_query = modality_text_map.get(image_type, "medical radiograph")
        inputs = siglip_processor(
            text=[text_query], return_tensors="pt", padding=True
        ).to(device)
        with torch.no_grad():
            outputs = siglip_model.get_text_features(**inputs)
            emb = outputs if torch.is_tensor(outputs) else getattr(outputs, "pooler_output", outputs[0])
        emb = emb / emb.norm(dim=-1, keepdim=True)
        query_vec = emb.cpu().float().numpy().astype("float32")
        query_type = "text"

    rel_path = patient_image.replace(KAGGLE_ROOT + "/", "")
    query_desc = description_lookup.get(rel_path, "No description.")
    results = faiss_retrieve(query_vec, k=k)

    return json.dumps({
        "query_image": patient_image, "query_description": query_desc,
        "image_type": image_type, "query_type": query_type,
        "k": len(results), "similar_images": results,
        "index_source": "chexpert_cxr",  # Flag for downstream awareness
    }, indent=2)


# ──── Tool 2: few shot image analysis — MODALITY ADAPTIVE ────────────────────
def _few_shot_image_analysis(knn_images: str, patient_image: str, body_part: str = "chest") -> str:
    """Analyze image using KNN examples as few-shot context.

    Adapts prompt and schema to the detected modality + body part.
    """
    try:
        knn_data = json.loads(knn_images) if isinstance(knn_images, str) else knn_images
        similar  = knn_data.get("similar_images", [])
    except (json.JSONDecodeError, TypeError):
        knn_data = {}
        similar = []

    # Extract image_type from KNN output (Tool 1 embeds it)
    image_type = knn_data.get("image_type", "xray")

    context_parts = []
    for r in similar[:5]:
        context_parts.append(f"- Similar case (score={r.get('score', 0)}): {r.get('description', '')}")
    context_str = "\n".join(context_parts) if context_parts else "No similar cases found."

    # ── Select modality-appropriate prompt ──
    analysis_prompt = get_analysis_prompt(image_type, body_part)

    # Adapt role description to modality
    role_map = {
        ("xray", "chest"): "expert radiologist performing chest X-ray analysis",
        ("ct", "head"): "expert neuroradiologist performing CT head analysis",
        ("ct", "brain"): "expert neuroradiologist performing CT head analysis",
        ("ct", "abdomen"): "expert abdominal radiologist performing CT abdomen analysis",
    }
    role = role_map.get((image_type, body_part), "expert radiologist performing medical image analysis")

    prompt = (
        f"You are an {role}.\n\n"
        f"SIMILAR CASES FROM DATABASE:\n{context_str}\n\n"
        f"{analysis_prompt}\n\n"
        f"Analyze the patient's image and return ONLY the JSON object."
    )

    content_blocks = [{"type": "text", "text": prompt}]
    if os.path.exists(patient_image):
        content_blocks.insert(0, {"type": "image", "image": Image.open(patient_image).convert("RGB")})

    response = run_medgemma(
        messages=[{"role": "user", "content": content_blocks}],
        max_new_tokens=512, temperature=0.1,
    )

    # ── Validate with modality-appropriate Pydantic schema ──
    schema_cls = get_review_schema(image_type, body_part)
    validated = extract_and_validate_json(response, schema_cls, run_medgemma)
    return json.dumps(validated)


# ──── Tool 3: verify analysis — MODALITY ADAPTIVE ────────────────────────────
def _verify_few_shot_image_analysis(analysis: str, patient_image: str, body_part: str = "chest") -> str:
    """Verify and correct the analysis with a second LLM pass + Pydantic validation.

    Adapts verification prompt and schema to the detected modality.
    """
    # Parse the preliminary analysis to detect modality
    try:
        analysis_data = json.loads(analysis) if isinstance(analysis, str) else analysis
    except (json.JSONDecodeError, TypeError):
        analysis_data = {}

    # Infer image_type from analysis content
    # CT Head fields are unique identifiers
    ct_head_fields = {"hemorrhage", "midline_shift", "hydrocephalus", "calvarial_fracture"}
    ct_abdo_fields = {"bowel_obstruction", "appendicitis", "hepatic_mass", "pneumoperitoneum"}

    analysis_keys = set(analysis_data.keys()) if isinstance(analysis_data, dict) else set()

    if analysis_keys & ct_head_fields:
        image_type = "ct"
        if body_part not in ("head", "brain"):
            body_part = "head"
    elif analysis_keys & ct_abdo_fields:
        image_type = "ct"
        if body_part != "abdomen":
            body_part = "abdomen"
    else:
        image_type = "xray"
        if body_part != "chest":
            body_part = "chest"

    # ── Select modality-appropriate prompt and schema ──
    analysis_prompt = get_analysis_prompt(image_type, body_part)
    schema_cls = get_review_schema(image_type, body_part)

    role_map = {
        ("xray", "chest"): "senior radiologist verifying a preliminary chest X-ray report",
        ("ct", "head"): "senior neuroradiologist verifying a preliminary CT head report",
        ("ct", "brain"): "senior neuroradiologist verifying a preliminary CT head report",
        ("ct", "abdomen"): "senior abdominal radiologist verifying a preliminary CT abdomen report",
    }
    role = role_map.get((image_type, body_part), "senior radiologist verifying a preliminary report")

    prompt = (
        f"You are a {role}.\n\n"
        f"PRELIMINARY ANALYSIS:\n{analysis}\n\n"
        f"Review the image again and correct any errors. "
        f"{analysis_prompt}"
    )

    content_blocks = [{"type": "text", "text": prompt}]
    if os.path.exists(patient_image):
        content_blocks.insert(0, {"type": "image", "image": Image.open(patient_image).convert("RGB")})

    response = run_medgemma(
        messages=[{"role": "user", "content": content_blocks}],
        max_new_tokens=512, temperature=0.1,
    )

    # ── Validate with modality-appropriate Pydantic schema + robust extraction ──
    validated = extract_and_validate_json(response, schema_cls, run_medgemma)
    return json.dumps(validated)


# ─── Tool 4: localize_abnormalities ───────────────────────────────────────────

def _localize_abnormalities(verified_analysis: str, patient_image: str) -> str:
    """Localize abnormalities on the patient image using MedGemma's
    bounding box prediction capability.

    Bounding box format: [y0, x0, y1, x1] normalized to [0, 1000]
    """
    try:
        analysis = json.loads(verified_analysis) if isinstance(verified_analysis, str) else verified_analysis
    except (json.JSONDecodeError, TypeError):
        analysis = {}

    # Extract positive / uncertain findings to localize
    findings_to_localize = []
    skip_keys = {"_meta", "verification_confidence", "verification_notes",
                 "view", "section", "other_abnormal_features", "other_findings"}
    for key, value in analysis.items():
        if key in skip_keys or key.startswith("_"):
            continue
        val_str = str(value).lower()
        if val_str in ("yes", "unclear", "true"):
            label = key.replace("_", " ").title()
            findings_to_localize.append(label)

    if not findings_to_localize:
        return json.dumps({
            "localized_image_path": patient_image,
            "bounding_boxes": [],
            "message": "No positive findings to localize.",
        })

    all_boxes = []
    for finding in findings_to_localize:
        localization_prompt = (
            'Instructions:\nThe following user query will require outputting '
            'bounding boxes. The format of bounding boxes coordinates is '
            '[y0, x0, y1, x1] where (y0, x0) must be top-left corner and '
            '(y1, x1) the bottom-right corner. Always normalize the x and y '
            'coordinates the range [0, 1000].\n'
            'You MUST output a single parseable json list of objects enclosed '
            'into ```json...``` brackets.\n\n'
            'Remember "left" refers to the patient\'s left side where the '
            'heart is.\n\n'
            f'Query:\nWhere is the {finding}? Output the final answer in the '
            'format "Final Answer: X" where X is a JSON list of objects with '
            '"box_2d" and "label" keys. Answer:'
        )

        content_blocks = [{"type": "text", "text": localization_prompt}]
        # Include actual image for real bounding box prediction
        if os.path.exists(patient_image):
            content_blocks.insert(0, {"type": "image", "image": Image.open(patient_image).convert("RGB")})

        response = run_medgemma(messages=[{"role": "user", "content": content_blocks}],
                                max_new_tokens=500, temperature=0.0)

        # Parse bounding boxes from response
        json_match = re.search(r"```json\s*(.*?)```", response, re.DOTALL)
        if json_match:
            try:
                boxes = json.loads(json_match.group(1).strip())
                if isinstance(boxes, list):
                    all_boxes.extend(boxes)
                    continue
            except json.JSONDecodeError:
                pass

        # Fallback: try extracting from "Final Answer:" pattern
        fa_match = re.search(r"Final Answer:\s*(\[.*?\])", response, re.DOTALL)
        if fa_match:
            try:
                boxes = json.loads(fa_match.group(1))
                if isinstance(boxes, list):
                    all_boxes.extend(boxes)
                    continue
            except json.JSONDecodeError:
                pass

        all_boxes.append({
            "box_2d": None,
            "label": finding,
            "note": "Localization unavailable — manual review recommended",
        })

    output_path = f"localized_{Path(patient_image).stem}_{uuid.uuid4().hex[:6]}.png"

    return json.dumps({
        "localized_image_path": output_path,
        "bounding_boxes": all_boxes,
        "num_findings_localized": len(findings_to_localize),
        "findings": findings_to_localize,
    })


# ─── Tool 5: retrieve_patient_previous_images ────────────────────────────────

def _retrieve_patient_previous_images(patient_id: str, image_type: str) -> str:
    """Retrieve a patient's prior imaging studies from the EHR/PACS."""
    patient = RADIOLOGY_EHR_DB.get(patient_id.upper())
    if not patient:
        return json.dumps({
            "patient_id": patient_id,
            "previous_images": [],
            "message": f"Patient {patient_id} not found in database.",
        })

    prior = patient.get("prior_imaging", [])
    filtered = [
        img for img in prior
        if img.get("type", "").lower() == image_type.lower()
    ]

    return json.dumps({
        "patient_id": patient_id,
        "patient_name": patient.get("name", "Unknown"),
        "num_prior_studies": len(filtered),
        "previous_images": filtered,
    })


# ─── Tool 6: run_longitudinal_review ─────────────────────────────────────────

def _run_longitudinal_review(
    patient_image: str, previous_images: str, verified_analysis: str
) -> str:
    """Compare current study against prior imaging for interval changes."""
    try:
        priors = json.loads(previous_images) if isinstance(previous_images, str) else previous_images
        analysis = json.loads(verified_analysis) if isinstance(verified_analysis, str) else verified_analysis
    except (json.JSONDecodeError, TypeError):
        priors = {"previous_images": []}
        analysis = {}

    prior_list = priors.get("previous_images", [])
    if not prior_list:
        return (
            "LONGITUDINAL REVIEW: No prior imaging available for comparison. "
            "This is the baseline study. Current findings documented in the "
            "verified analysis should be used as the reference point for "
            "future comparisons."
        )

    prior_summaries = "\n".join(
        f"  - {p.get('date', 'Unknown date')} [{p.get('type', '?')}]: "
        f"{p.get('report', 'No report')}"
        for p in prior_list
    )

    current_findings = "\n".join(
        f"  {k}: {v}" for k, v in analysis.items()
        if not k.startswith("_") and v is not None
    )

    prompt = (
        f"You are a Senior Radiologist performing a longitudinal comparison.\n\n"
        f"CURRENT STUDY FINDINGS:\n{current_findings}\n\n"
        f"PRIOR STUDIES:\n{prior_summaries}\n\n"
        f"Provide a structured comparison:\n"
        f"1. NEW FINDINGS: What is seen now but not before?\n"
        f"2. RESOLVED FINDINGS: What was seen before but not now?\n"
        f"3. PROGRESSING/WORSENING: What has gotten worse?\n"
        f"4. STABLE FINDINGS: What is unchanged?\n"
        f"5. OVERALL INTERVAL CHANGE ASSESSMENT: "
        f"[Improved / Stable / Worsened / Mixed]\n"
    )

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    review = run_medgemma(messages=messages, max_new_tokens=512, temperature=0.2)
    return review


# ─── Tool 7: revise_report ───────────────────────────────────────────────────

def _revise_report(
    clinician_critique: str,
    verified_analysis: str,
    longitudinal_review: str = None,
) -> str:
    """Revise the radiology report based on clinician feedback."""
    try:
        analysis = json.loads(verified_analysis) if isinstance(verified_analysis, str) else verified_analysis
    except (json.JSONDecodeError, TypeError):
        analysis = {}

    context_parts = [f"ORIGINAL ANALYSIS:\n{json.dumps(analysis, indent=2)}"]
    if longitudinal_review:
        context_parts.append(f"\nLONGITUDINAL REVIEW:\n{longitudinal_review}")

    prompt = (
        f"You are a Radiology report editor. A supervising clinician has "
        f"provided feedback on the draft report. Incorporate their critique "
        f"while maintaining clinical accuracy.\n\n"
        f"{''.join(context_parts)}\n\n"
        f"CLINICIAN CRITIQUE:\n{clinician_critique}\n\n"
        f"Generate the REVISED REPORT. Maintain all accurate original "
        f"findings, incorporate the clinician's corrections, and flag any "
        f"remaining uncertainties. Format as a standard radiology report with "
        f"sections: INDICATION, TECHNIQUE, FINDINGS, IMPRESSION."
    )

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    revised = run_medgemma(messages=messages, max_new_tokens=512, temperature=0.2)
    return revised


# ─── Tool 8: retrieve_ehr ────────────────────────────────────────────────────

def _retrieve_ehr(patient_id: str) -> str:
    """Retrieve the patient's Electronic Health Record."""
    patient = RADIOLOGY_EHR_DB.get(patient_id.upper())
    if not patient:
        return (
            f"No EHR records found for patient {patient_id}. "
            f"Please verify the patient ID."
        )

    record = (
        f"PATIENT: {patient.get('name', 'Unknown')} | "
        f"Age: {patient.get('age', '?')} | Sex: {patient.get('sex', '?')}\n"
        f"CHIEF COMPLAINT: {patient.get('complaints', 'Not documented')}\n"
        f"CLINICAL NOTES: {patient.get('clinical_notes', 'None')}\n"
        f"CURRENT MEDICATIONS: {patient.get('prescriptions', 'None')}\n"
        f"ALLERGIES: {', '.join(patient.get('allergies', [])) or 'NKDA'}\n"
        f"PRIOR IMAGING: {len(patient.get('prior_imaging', []))} studies on file"
    )
    return record


# ─── Tool 9: generate_soap_report ────────────────────────────────────────────

def _generate_soap_report(
    medical_records: str,
    verified_analysis: str,
    updated_report: str = None,
    longitudinal_review: str = None,
) -> str:
    """Generate a SOAP note integrating EHR data and radiology findings."""
    radiology_data = updated_report or verified_analysis

    context_parts = [
        f"PATIENT RECORDS:\n{medical_records}",
        f"\nRADIOLOGY FINDINGS:\n{radiology_data}",
    ]
    if longitudinal_review:
        context_parts.append(f"\nLONGITUDINAL REVIEW:\n{longitudinal_review}")

    prompt = (
        f"Generate a comprehensive SOAP note for a radiology consultation.\n\n"
        f"{''.join(context_parts)}\n\n"
        f"Format:\n"
        f"SUBJECTIVE: Patient's presenting complaints and relevant history\n"
        f"OBJECTIVE: Imaging findings (from the radiology analysis), vitals, labs\n"
        f"ASSESSMENT: Clinical interpretation integrating history + imaging\n"
        f"PLAN: Recommended next steps, follow-up imaging, referrals\n\n"
        f"Be concise but thorough. Flag any critical findings requiring "
        f"immediate action."
    )

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    soap = run_medgemma(messages=messages, max_new_tokens=512, temperature=0.2)
    return soap


# ─── Tool 10: build_pdf_soap_report ──────────────────────────────────────────

def _build_pdf_soap_report_v0(
    soap_report: str, localized_image: str, patient_image: str
) -> str:
    """Build a PDF document containing the SOAP report and annotated image."""
    try:
        loc_data = json.loads(localized_image) if isinstance(localized_image, str) else localized_image
    except (json.JSONDecodeError, TypeError):
        loc_data = {}

    localized_path = loc_data.get("localized_image_path", patient_image)
    bboxes = loc_data.get("bounding_boxes", [])

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_filename = f"radiology_report_{timestamp}.pdf"

    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.units import mm
        from reportlab.platypus import (
            SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
        )
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

        doc = SimpleDocTemplate(pdf_filename, pagesize=A4)
        styles = getSampleStyleSheet()

        title_style = ParagraphStyle(
            "ReportTitle", parent=styles["Title"], fontSize=16, spaceAfter=12,
        )
        heading_style = ParagraphStyle(
            "SectionHead", parent=styles["Heading2"], fontSize=12,
            spaceAfter=6, spaceBefore=12,
        )
        body_style = ParagraphStyle(
            "ReportBody", parent=styles["Normal"], fontSize=10,
            spaceAfter=4, leading=14,
        )

        story = []
        story.append(Paragraph("RADIOLOGY CONSULTATION REPORT", title_style))
        story.append(Paragraph(
            f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
            body_style,
        ))
        story.append(Spacer(1, 12))

        for line in soap_report.split("\n"):
            line = line.strip()
            if not line:
                story.append(Spacer(1, 6))
            elif line.startswith(("SUBJECTIVE", "OBJECTIVE", "ASSESSMENT", "PLAN")):
                story.append(Paragraph(line, heading_style))
            else:
                safe = line.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
                story.append(Paragraph(safe, body_style))

        if bboxes:
            story.append(Spacer(1, 12))
            story.append(Paragraph("LOCALIZED FINDINGS", heading_style))
            for box in bboxes:
                label = box.get("label", "Unknown")
                coords = box.get("box_2d", "N/A")
                story.append(Paragraph(f"* {label}: coordinates {coords}", body_style))

        if os.path.exists(localized_path):
            story.append(Spacer(1, 12))
            story.append(Paragraph("ANNOTATED IMAGE", heading_style))
            story.append(RLImage(localized_path, width=150 * mm, height=150 * mm))

        doc.build(story)

    except ImportError:
        with open(pdf_filename, "w") as f:
            f.write("=" * 60 + "\n")
            f.write("  RADIOLOGY CONSULTATION REPORT\n")
            f.write(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
            f.write("=" * 60 + "\n\n")
            f.write(soap_report)
            f.write("\n\n" + "-" * 40 + "\n")
            f.write("LOCALIZED FINDINGS\n")
            for box in bboxes:
                f.write(f"  {box.get('label', '?')}: {box.get('box_2d', 'N/A')}\n")
            f.write(f"\nImages: {patient_image}, {localized_path}\n")

    return json.dumps({
        "generated_pdf_path": pdf_filename,
        "includes_localization": len(bboxes) > 0,
        "timestamp": datetime.now().isoformat(),
    })


def _build_pdf_soap_report(
    soap_report: str, localized_image: str, patient_image: str
) -> str:
    """
    Generate a beautifully formatted PDF with:
    - Proper headings (SUBJECTIVE, OBJECTIVE, etc.) in bold blue.
    - Clean body text with proper line spacing.
    - Annotated image (if available) scaled to fit.
    - Page numbers and timestamp.
    """
    try:
        loc_data = json.loads(localized_image) if isinstance(localized_image, str) else localized_image
    except (json.JSONDecodeError, TypeError):
        loc_data = {}

    localized_path = loc_data.get("localized_image_path", patient_image)
    bboxes = loc_data.get("bounding_boxes", [])

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_filename = f"radiology_report_{timestamp}.pdf"

    # ------------------------------------------------------------
    # 1. Try ReportLab – if it fails, fall back to plain text
    # ------------------------------------------------------------
    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.units import mm, inch
        from reportlab.platypus import (
            SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
            PageBreak
        )
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib import colors
        from reportlab.lib.enums import TA_CENTER, TA_LEFT
        from reportlab.pdfbase import pdfmetrics
        from reportlab.pdfbase.ttfonts import TTFont

        # Optional: try to register a nicer font
        try:
            pdfmetrics.registerFont(TTFont('DejaVuSans', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'))
            pdfmetrics.registerFont(TTFont('DejaVuSans-Bold', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))
            default_font = 'DejaVuSans'
        except:
            default_font = 'Helvetica'

        doc = SimpleDocTemplate(pdf_filename, pagesize=A4,
                                leftMargin=inch, rightMargin=inch,
                                topMargin=inch, bottomMargin=inch)
        styles = getSampleStyleSheet()

        # Custom styles
        title_style = ParagraphStyle(
            'ReportTitle',
            parent=styles['Title'],
            fontName=default_font,
            fontSize=20,
            leading=24,
            alignment=TA_CENTER,
            spaceAfter=12,
            textColor=colors.HexColor('#0a2472')
        )
        heading_style = ParagraphStyle(
            'SectionHead',
            parent=styles['Heading2'],
            fontName=default_font,
            fontSize=14,
            leading=18,
            spaceBefore=12,
            spaceAfter=6,
            textColor=colors.HexColor('#0a2472'),
            borderWidth=0,
        )
        subheading_style = ParagraphStyle(
            'SubHead',
            parent=styles['Heading3'],
            fontName=default_font,
            fontSize=12,
            leading=15,
            spaceBefore=8,
            spaceAfter=4,
            textColor=colors.HexColor('#1e3a8a'),
        )
        body_style = ParagraphStyle(
            'ReportBody',
            parent=styles['Normal'],
            fontName=default_font,
            fontSize=11,
            leading=14,
            spaceAfter=4,
        )
        bullet_style = ParagraphStyle(
            'BulletList',
            parent=body_style,
            leftIndent=20,
            bulletIndent=10,
            spaceAfter=2,
        )
        footer_style = ParagraphStyle(
            'Footer',
            parent=styles['Normal'],
            fontName=default_font,
            fontSize=8,
            leading=10,
            textColor=colors.grey,
            alignment=TA_CENTER,
        )

        story = []

        # --- Title and timestamp ---
        story.append(Paragraph("RADIOLOGY CONSULTATION REPORT", title_style))
        story.append(Paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}", body_style))
        story.append(Spacer(1, 12))

        # --- Parse SOAP text and convert to styled paragraphs ---
        # Heuristics:
        # - Lines that are ALL CAPS and end with ':' are main headings (e.g., "SUBJECTIVE:")
        # - Lines wrapped in ** ** are also headings (remove the asterisks)
        # - Lines starting with a number + '.' are subheadings (e.g., "1. Recommendation")
        # - Bullet lines starting with '*', '-', or '•' become bullet lists

        lines = soap_report.split('\n')
        i = 0
        while i < len(lines):
            raw_line = lines[i].rstrip()
            if not raw_line:
                story.append(Spacer(1, 6))
                i += 1
                continue

            # Remove common markdown artifacts
            line = raw_line.replace('**', '').replace('__', '').strip()

            # Detect main heading (all caps with colon)
            if line.upper() == line and ':' in line:
                story.append(Paragraph(line, heading_style))
                i += 1
                continue

            # Detect subheading (numbered, e.g., "1. Recommendation")
            if re.match(r'^\d+\.\s+\w+', line):
                story.append(Paragraph(line, subheading_style))
                i += 1
                continue

            # Detect bullet list items
            if line.startswith(('* ', '- ', '• ')):
                # Remove the bullet character, let ReportLab add its own
                text = line[2:].strip()
                story.append(Paragraph(f'• {text}', bullet_style))
                i += 1
                continue

            # Otherwise, treat as normal paragraph
            story.append(Paragraph(line, body_style))
            i += 1

        # --- Localized findings (if any) ---
        if bboxes and any(b.get('box_2d') for b in bboxes):
            story.append(Spacer(1, 12))
            story.append(Paragraph("LOCALIZED FINDINGS", heading_style))
            for box in bboxes:
                label = box.get('label', 'Unknown')
                coords = box.get('box_2d', 'N/A')
                note = box.get('note', '')
                text = f"• <b>{label}</b>: {coords}"
                if note:
                    text += f" <i>({note})</i>"
                story.append(Paragraph(text, bullet_style))

        # --- Annotated image ---
        if os.path.exists(localized_path):
            story.append(Spacer(1, 12))
            story.append(Paragraph("ANNOTATED IMAGE", heading_style))
            img = RLImage(localized_path, width=150*mm, height=150*mm, kind='proportional')
            story.append(img)
            story.append(Spacer(1, 6))
            story.append(Paragraph("Figure: Annotated study image with bounding boxes.", body_style))

        # --- Footer with page numbers ---
        def add_page_number(canvas, doc):
            canvas.saveState()
            canvas.setFont(default_font, 8)
            canvas.setFillColor(colors.grey)
            page_num = f"Page {doc.page}"
            canvas.drawRightString(doc.width + doc.leftMargin - 6*mm, doc.bottomMargin - 10, page_num)
            canvas.restoreState()

        doc.build(story, onLaterPages=add_page_number, onFirstPage=add_page_number)

    except Exception as e:
        # --------------------------------------------------------
        # 2. Fallback: plain text (only if ReportLab truly fails)
        # --------------------------------------------------------
        with open(pdf_filename, "w") as f:
            f.write("=" * 60 + "\n")
            f.write("  RADIOLOGY CONSULTATION REPORT\n")
            f.write(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
            f.write("=" * 60 + "\n\n")
            # Clean up the text: remove markdown asterisks
            clean_text = soap_report.replace('**', '').replace('__', '')
            f.write(clean_text)
            f.write("\n\n" + "-" * 40 + "\n")
            f.write("LOCALIZED FINDINGS\n")
            for box in bboxes:
                f.write(f"  {box.get('label', '?')}: {box.get('box_2d', 'N/A')}\n")
            f.write(f"\nImages: {patient_image}, {localized_path}\n")

    return json.dumps({
        "generated_pdf_path": pdf_filename,
        "includes_localization": len(bboxes) > 0,
        "timestamp": datetime.now().isoformat(),
    })

# ─── Tool 11: retrieve_session_memory ────────────────────────────────────────

def _retrieve_session_memory(session_id: str) -> str:
    """Retrieve accumulated context from a prior session."""
    return SESSION_STORE.get_session_summary(session_id)


# ─── Tool 12: qa_followup ───────────────────────────────────────────────────

def _qa_followup(followup_question: str, session_context: str) -> str:
    """Answer a follow-up question using accumulated session context."""
    prompt = (
        f"You are an expert Co-Radiologist answering a follow-up question "
        f"about a radiology case you previously reviewed.\n\n"
        f"SESSION CONTEXT (prior analysis data):\n{session_context}\n\n"
        f"CLINICIAN'S FOLLOW-UP QUESTION: {followup_question}\n\n"
        f"Answer the question using ONLY information from the session context. "
        f"If the information needed is not available in context, say so clearly "
        f"and suggest what additional analysis might be needed. "
        f"Be concise and clinically precise."
    )

    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    answer = run_medgemma(messages=messages, max_new_tokens=512, temperature=0.2)
    return answer


# ─── Tool 13: store_session_memory ───────────────────────────────────────────

def _store_session_memory(session_id: str, data_to_store: str) -> str:
    """Persist analysis results into the session store."""
    SESSION_STORE.save_context(session_id, f"saved_{datetime.now().isoformat()}", data_to_store)
    return f"Session {session_id}: data stored successfully."


# ─── Tool 14: classify_image_modality ────────────────────────────────────────

def _classify_image_modality(patient_image: str) -> str:
    """Auto-detect imaging modality and body region from the image."""
    prompt = (
        f"Identify the medical imaging modality and body region.\n\n"
        f"Return ONLY a JSON object with:\n"
        f'{{"image_type": "xray|ct|mri", "body_part": "chest|head|abdomen|brain|other"}}'
    )

    content_blocks = [{"type": "text", "text": prompt}]
    # Include actual image for real classification
    if os.path.exists(patient_image):
        content_blocks.insert(0, {"type": "image", "image": Image.open(patient_image).convert("RGB")})

    messages = [{"role": "user", "content": content_blocks}]
    response = run_medgemma(messages=messages, max_new_tokens=100, temperature=0.0)
    parsed = extract_and_validate_json(response)

    if "image_type" not in parsed:
        parsed["image_type"] = "xray"
    if "body_part" not in parsed:
        parsed["body_part"] = "chest"

    return json.dumps(parsed)

# ─── Tool 15: synthesize_clinical_narrative ───────────────────────────────────

def _synthesize_clinical_narrative(verified_analysis: str, patient_image: str, body_part: str = "chest") -> str:
    """Convert structured JSON findings into concise clinical prose
    written in the style of an attending radiologist's dictated report."""

    try:
        analysis = json.loads(verified_analysis) if isinstance(verified_analysis, str) else verified_analysis
    except (json.JSONDecodeError, TypeError):
        analysis = {}

    # Separate positive, negative, and uncertain findings
    positives, negatives, uncertains = [], [], []
    skip = {"_meta", "verification_confidence", "verification_notes",
            "view", "section", "other_abnormal_features"}

    for k, v in analysis.items():
        if k in skip or k.startswith("_"):
            continue
        label = k.replace("_", " ")
        val = str(v).lower()
        if val == "yes":
            positives.append(label)
        elif val == "unclear":
            uncertains.append(label)
        else:
            negatives.append(label)

    # Build a compact summary for the prompt (avoids feeding raw JSON)
    findings_summary = json.dumps({
        "positive": positives,
        "uncertain": uncertains,
        "negative": negatives,
        "confidence": analysis.get("verification_confidence", "medium"),
        "notes": analysis.get("verification_notes", ""),
    }, indent=2)

    prompt = f"""You are a board-certified radiologist dictating a report.
Write a FINDINGS and IMPRESSION section for this {body_part} study.

STRUCTURED ANALYSIS:
{findings_summary}

RULES — follow these exactly:
1. Write in standard radiology report prose — short declarative sentences, present tense.
2. FINDINGS: Describe systematically by anatomic region. State positive findings first, then pertinent negatives only. Do NOT exhaustively list every negative.
3. IMPRESSION: 2-4 numbered key takeaways, most critical first. If no abnormality, state "No acute cardiopulmonary process" or equivalent.
4. Use precise radiologic language (e.g. "opacity" not "shadow", "cardiomegaly" not "big heart").
5. Total length: 80-150 words. Do NOT pad, repeat, or restate.
6. No JSON, no markdown, no headers beyond "FINDINGS:" and "IMPRESSION:".
7. Stop immediately when the clinical content is fully conveyed.

EXAMPLE (chest X-ray with cardiomegaly + effusion):
FINDINGS: The cardiac silhouette is mildly enlarged. There is a small left-sided pleural effusion with meniscus sign. The lungs are otherwise clear without focal consolidation or pneumothorax. No acute osseous abnormality. Support devices are not identified.

IMPRESSION:
1. Mild cardiomegaly.
2. Small left pleural effusion. Clinical correlation recommended.

EXAMPLE (normal chest X-ray):
FINDINGS: The cardiac silhouette is normal in size. The mediastinal contours are unremarkable. The lungs are clear bilaterally without focal opacity, effusion, or pneumothorax. No acute osseous abnormality.

IMPRESSION:
1. No acute cardiopulmonary process.

Now write the report for the current study:"""

    content_blocks = [{"type": "text", "text": prompt}]
    if os.path.exists(patient_image):
        content_blocks.insert(0, {"type": "image", "image": Image.open(patient_image).convert("RGB")})

    response = run_medgemma(
        messages=[{"role": "user", "content": content_blocks}],
        max_new_tokens=384,
        temperature=0.2,
    )

    # Safety net: if model repeats FINDINGS section, keep only the first
    sections = response.split("FINDINGS:")
    if len(sections) > 2:
        response = "FINDINGS:" + sections[1]

    return response.strip()


# ─── Legacy alias for get_prompt_skill (used by old code) ────────────────────
def get_prompt_skill(image_type: str, body_part: str = "chest") -> str:
    """Alias for get_analysis_prompt — backward compat."""
    return get_analysis_prompt(image_type, body_part)


print("✓ All 15 tool functions defined (modality-adaptive)")

✓ All 15 tool functions defined (modality-adaptive)


## **Co-Radiology Agent: Tool Registration**


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
#  TOOL REGISTRATION — nanoathens v0.2.0 Schema
#  Fixed: Tools 2+3 now receive body_part for modality-adaptive prompts
# ══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry = ToolRegistry()

# ── Shared Extractors ─────────────────────────────────────────────────────────
_PATIENT_ID_EXTRACTOR = (
    ArgExtractorType.ALPHANUMERIC_ID,
    {"pattern": r"RAD\d{3}", "preceding_words": ["patient", "id", "rad"]},
)

_SESSION_ID_EXTRACTOR = (
    ArgExtractorType.ALPHANUMERIC_ID,
    {"pattern": r"RAD-[A-Z0-9]{8}", "preceding_words": ["session"]},
)

_IMAGE_TYPE_EXTRACTOR = (
    ArgExtractorType.ENUM, {"values": ["xray", "ct", "mri"]},
)

_BODY_PART_EXTRACTOR = (
    ArgExtractorType.ENUM, {"values": ["chest", "head", "abdomen", "brain", "other"]},
)

_QUOTED_EXTRACTOR = (ArgExtractorType.QUOTED, {})


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 1: retrieve_similar_images
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="retrieve_similar_images",
    description="Retrieve similar medical images from FAISS index using MedSigLIP embeddings",
    parameters={
        "patient_image": "Path to the patient image file",
        "image_type": "Imaging modality: xray|ct|mri",
    },
    required=["patient_image", "image_type"],
    example={
        "patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg",
        "image_type": "xray",
    },
    docstring="Embeds query image with MedSigLIP-448, searches FAISS index, returns top-K similar cases with cosine similarity scores and CheXpert label descriptions.",
    tool_type=ToolType.RETRIEVAL,
    func=_retrieve_similar_images,
    arg_sources={"patient_image": "patient_image_input", "image_type": "image_type_input"},
    output_keys={"knn_images": "JSON array of similar images with scores and descriptions"},
    explicit_keywords=["similar", "retrieve", "knn", "search", "faiss", "neighbors"],
    arg_extractors={
        "patient_image": _QUOTED_EXTRACTOR,
        "image_type": _IMAGE_TYPE_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 2: few_shot_image_analysis  (+ body_part for adaptive prompts)
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="few_shot_image_analysis",
    description="Analyze medical image using retrieved similar images as few-shot context (adapts to CXR/CT Head/CT Abdomen)",
    parameters={
        "knn_images": "JSON string of KNN retrieval results from retrieve_similar_images",
        "patient_image": "Path to the patient image file",
        "body_part": "Body region: chest|head|abdomen|brain|other",
    },
    required=["knn_images", "patient_image", "body_part"],
    example={
        "knn_images": '{"similar_images": [{"path": "...", "score": 0.89}]}',
        "patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg",
        "body_part": "chest",
    },
    docstring="Uses top-K similar cases as few-shot context for MedGemma analysis. Adapts prompt and Pydantic schema to modality: CXR (14 CheXpert labels), CT Head (12 neuro findings), CT Abdomen (15 abdominal findings).",
    tool_type=ToolType.COMPUTATION,
    func=_few_shot_image_analysis,
    arg_sources={
        "knn_images": "knn_images",
        "patient_image": "patient_image_input",
        "body_part": "body_part_input",
    },
    output_keys={"few_shot_analysis": "Preliminary analysis JSON string (modality-specific)"},
    explicit_keywords=["analyze", "analysis", "few-shot", "diagnose", "findings"],
    arg_extractors={
        "knn_images": _QUOTED_EXTRACTOR,
        "patient_image": _QUOTED_EXTRACTOR,
        "body_part": _BODY_PART_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 3: verify_few_shot_image_analysis  (+ body_part)
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="verify_few_shot_image_analysis",
    description="Verify and validate the analysis with a second LLM pass and Pydantic schema enforcement (adapts to modality)",
    parameters={
        "analysis": "Preliminary analysis string from few_shot_image_analysis",
        "patient_image": "Path to the patient image file",
        "body_part": "Body region: chest|head|abdomen|brain|other",
    },
    required=["analysis", "patient_image", "body_part"],
    example={
        "analysis": '{"no_finding": "no", "cardiomegaly": "yes", "lung_opacity": "no"}',
        "patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg",
        "body_part": "chest",
    },
    docstring="Senior radiologist verification pass. Re-examines the image, corrects errors in the preliminary analysis, and enforces modality-specific Pydantic schema (CXRReview / CTHeadReview / CTAbdominalReview).",
    tool_type=ToolType.COMPUTATION,
    func=_verify_few_shot_image_analysis,
    arg_sources={
        "analysis": "few_shot_analysis",
        "patient_image": "patient_image_input",
        "body_part": "body_part_input",
    },
    output_keys={"verified_analysis": "Validated analysis JSON (Pydantic-enforced, modality-specific)"},
    explicit_keywords=["verify", "validate", "check", "review", "confirm", "pydantic"],
    arg_extractors={
        "analysis": _QUOTED_EXTRACTOR,
        "patient_image": _QUOTED_EXTRACTOR,
        "body_part": _BODY_PART_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 4: localize_abnormalities
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="localize_abnormalities",
    description="Localize abnormalities on the image using MedGemma bounding box prediction",
    parameters={
        "verified_analysis": "Validated analysis JSON from verify_few_shot_image_analysis",
        "patient_image": "Path to the patient image file",
    },
    required=["verified_analysis", "patient_image"],
    example={
        "verified_analysis": '{"cardiomegaly": "yes", "pleural_effusion": "yes"}',
        "patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg",
    },
    docstring="Extracts positive findings from verified analysis, prompts MedGemma for bounding box coordinates [y0, x0, y1, x1] normalized to [0, 1000], and returns annotated image path with localization data.",
    tool_type=ToolType.COMPUTATION,
    func=_localize_abnormalities,
    arg_sources={"verified_analysis": "verified_analysis", "patient_image": "patient_image_input"},
    output_keys={"localized_image": "JSON with localized_image_path and bounding_boxes array"},
    explicit_keywords=["localize", "locate", "bounding", "box", "annotate", "highlight", "region"],
    arg_extractors={
        "verified_analysis": _QUOTED_EXTRACTOR,
        "patient_image": _QUOTED_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 5: retrieve_patient_previous_images
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="retrieve_patient_previous_images",
    description="Retrieve a patient's prior imaging studies from EHR/PACS",
    parameters={
        "patient_id": "Patient identifier (e.g. RAD001)",
        "image_type": "Filter by modality: xray|ct|mri",
    },
    required=["patient_id", "image_type"],
    example={"patient_id": "RAD001", "image_type": "xray"},
    docstring="Looks up the patient in RADIOLOGY_EHR_DB, filters prior imaging by modality, returns JSON with study dates, reports, and image paths for longitudinal comparison.",
    tool_type=ToolType.RETRIEVAL,
    func=_retrieve_patient_previous_images,
    arg_sources={"patient_id": "patient_id_input", "image_type": "image_type_input"},
    output_keys={"previous_images": "JSON array of prior imaging studies"},
    explicit_keywords=["previous", "prior", "history", "past", "imaging", "pacs"],
    arg_extractors={
        "patient_id": _PATIENT_ID_EXTRACTOR,
        "image_type": _IMAGE_TYPE_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 6: run_longitudinal_review
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="run_longitudinal_review",
    description="Compare current study against prior imaging to identify interval changes",
    parameters={
        "patient_image": "Path to the current patient image",
        "previous_images": "JSON from retrieve_patient_previous_images",
        "verified_analysis": "JSON from verify_few_shot_image_analysis",
    },
    required=["patient_image", "previous_images", "verified_analysis"],
    example={
        "patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg",
        "previous_images": '{"previous_images": [{"date": "2025-01-15", "type": "xray", "report": "..."}]}',
        "verified_analysis": '{"cardiomegaly": "yes", "pleural_effusion": "no"}',
    },
    docstring="Senior radiologist longitudinal comparison. Identifies new findings, resolved findings, progressing disease, and stable findings. Returns structured interval change assessment.",
    tool_type=ToolType.COMPUTATION,
    func=_run_longitudinal_review,
    arg_sources={
        "patient_image": "patient_image_input",
        "previous_images": "previous_images",
        "verified_analysis": "verified_analysis",
    },
    output_keys={"longitudinal_review": "Structured longitudinal comparison narrative"},
    explicit_keywords=["longitudinal", "compare", "interval", "change", "progression", "prior"],
    arg_extractors={
        "patient_image": _QUOTED_EXTRACTOR,
        "previous_images": _QUOTED_EXTRACTOR,
        "verified_analysis": _QUOTED_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 7: revise_report
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="revise_report",
    description="Revise the radiology report based on clinician feedback and critique",
    parameters={
        "clinician_critique": "The clinician's feedback text",
        "verified_analysis": "JSON from verify_few_shot_image_analysis",
        "longitudinal_review": "Optional longitudinal comparison text",
    },
    required=["clinician_critique", "verified_analysis"],
    example={
        "clinician_critique": "Please re-evaluate the right lower lobe opacity",
        "verified_analysis": '{"lung_opacity": "yes", "consolidation": "unclear"}',
    },
    docstring="Radiology report editor. Incorporates clinician corrections while maintaining clinical accuracy. Outputs revised report with INDICATION, TECHNIQUE, FINDINGS, IMPRESSION sections.",
    tool_type=ToolType.COMPUTATION,
    func=_revise_report,
    arg_sources={
        "clinician_critique": "clinician_critique_input",
        "verified_analysis": "verified_analysis",
        "longitudinal_review": "longitudinal_review",
    },
    output_keys={"updated_report": "Revised radiology report text"},
    explicit_keywords=["revise", "edit", "correct", "feedback", "critique", "update", "report"],
    arg_extractors={
        "clinician_critique": _QUOTED_EXTRACTOR,
        "verified_analysis": _QUOTED_EXTRACTOR,
        "longitudinal_review": _QUOTED_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 8: retrieve_ehr
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="retrieve_ehr",
    description="Retrieve the patient's Electronic Health Record including complaints, notes, prescriptions, and allergies",
    parameters={
        "patient_id": "Patient identifier (e.g. RAD001)",
    },
    required=["patient_id"],
    example={"patient_id": "RAD001"},
    docstring="Retrieves structured EHR data from RADIOLOGY_EHR_DB including demographics, chief complaint, clinical notes, current medications, allergies, and prior imaging count.",
    tool_type=ToolType.RETRIEVAL,
    func=_retrieve_ehr,
    arg_sources={"patient_id": "patient_id_input"},
    output_keys={"medical_records": "Structured text of patient medical records"},
    explicit_keywords=["ehr", "medical", "records", "patient", "history", "chart", "clinical"],
    arg_extractors={
        "patient_id": _PATIENT_ID_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 9: generate_soap_report
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="generate_soap_report",
    description="Generate a SOAP note integrating EHR data and radiology findings",
    parameters={
        "medical_records": "Text from retrieve_ehr",
        "verified_analysis": "JSON from verify_few_shot_image_analysis",
    },
    required=["medical_records", "verified_analysis"],
    example={
        "medical_records": "PATIENT: John Doe | Age: 65 | CHIEF COMPLAINT: Shortness of breath",
        "verified_analysis": '{"cardiomegaly": "yes", "pleural_effusion": "yes"}',
    },
    docstring="Generates comprehensive SOAP (Subjective, Objective, Assessment, Plan) note for radiology consultation. Integrates EHR history with imaging findings. Flags critical findings requiring immediate action.",
    tool_type=ToolType.COMPUTATION,
    func=_generate_soap_report,
    arg_sources={
        "medical_records": "medical_records",
        "verified_analysis": "verified_analysis",
    },
    output_keys={"soap_report": "Complete SOAP note text"},
    explicit_keywords=["soap", "report", "note", "subjective", "objective", "assessment", "plan"],
    arg_extractors={
        "medical_records": _QUOTED_EXTRACTOR,
        "verified_analysis": _QUOTED_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 10: build_pdf_soap_report
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="build_pdf_soap_report",
    description="Build a PDF document containing the SOAP report and annotated image",
    parameters={
        "soap_report": "SOAP note text from generate_soap_report",
        "localized_image": "JSON from localize_abnormalities with image path and bounding boxes",
        "patient_image": "Original patient image path",
    },
    required=["soap_report", "localized_image", "patient_image"],
    example={
        "soap_report": "SUBJECTIVE: 65yo male with SOB...",
        "localized_image": '{"localized_image_path": "localized_001.png", "bounding_boxes": []}',
        "patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg",
    },
    docstring="Builds a professional PDF radiology report using ReportLab. Includes SOAP sections, localization bounding box data, and annotated CXR image. Falls back to text-based PDF if ReportLab unavailable.",
    tool_type=ToolType.COMPUTATION,
    func=_build_pdf_soap_report,
    arg_sources={
        "soap_report": "soap_report",
        "localized_image": "localized_image",
        "patient_image": "patient_image_input",
    },
    output_keys={"generated_pdf_path": "File path to the generated PDF report"},
    explicit_keywords=["pdf", "document", "report", "generate", "build", "export", "download"],
    arg_extractors={
        "soap_report": _QUOTED_EXTRACTOR,
        "localized_image": _QUOTED_EXTRACTOR,
        "patient_image": _QUOTED_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 11: retrieve_session_memory
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="retrieve_session_memory",
    description="Retrieve accumulated context from a prior session for follow-up queries",
    parameters={
        "session_id": "Session identifier (e.g. RAD-A1B2C3D4)",
    },
    required=["session_id"],
    example={"session_id": "RAD-A1B2C3D4"},
    docstring="Retrieves full session state from SessionStore including conversation history, accumulated context keys, and prior tool outputs. Enables multi-turn interaction without re-running the pipeline.",
    tool_type=ToolType.RETRIEVAL,
    func=_retrieve_session_memory,
    arg_sources={"session_id": "session_id_input"},
    output_keys={"session_context": "Text summary of all prior context in this session"},
    explicit_keywords=["session", "memory", "context", "history", "previous", "prior", "recall"],
    arg_extractors={
        "session_id": _SESSION_ID_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 12: qa_followup
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="qa_followup",
    description="Answer a follow-up question about a previous radiology analysis using session context",
    parameters={
        "followup_question": "The clinician's follow-up question",
        "session_context": "Text summary from retrieve_session_memory",
    },
    required=["followup_question", "session_context"],
    example={
        "followup_question": "Is the cardiomegaly getting worse compared to the prior study?",
        "session_context": "Session RAD-A1B2C3D4: verified_analysis shows cardiomegaly=yes...",
    },
    docstring="Conversational follow-up tool. Answers clinician questions using accumulated session context without re-running the full imaging pipeline. Responds only from available context and flags information gaps.",
    tool_type=ToolType.KNOWLEDGE,
    func=_qa_followup,
    arg_sources={
        "followup_question": "followup_question_input",
        "session_context": "session_context",
    },
    output_keys={"qa_response": "Clinically grounded answer to the follow-up question"},
    explicit_keywords=["question", "followup", "ask", "clarify", "explain", "why", "what"],
    arg_extractors={
        "followup_question": _QUOTED_EXTRACTOR,
        "session_context": _QUOTED_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 13: store_session_memory
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="store_session_memory",
    description="Persist analysis results into the session store for future follow-up queries",
    parameters={
        "session_id": "Session identifier (e.g. RAD-A1B2C3D4)",
        "data_to_store": "Stringified data to persist",
    },
    required=["session_id", "data_to_store"],
    example={
        "session_id": "RAD-A1B2C3D4",
        "data_to_store": '{"verified_analysis": {...}, "soap_report": "..."}',
    },
    docstring="Writes analysis data to the in-memory SessionStore keyed by session_id. Enables multi-turn workflows where follow-up queries can access prior results without re-computation.",
    tool_type=ToolType.COMPUTATION,
    func=_store_session_memory,
    arg_sources={
        "session_id": "session_id_input",
        "data_to_store": "data_to_store_input",
    },
    output_keys={"session_save_confirmation": "Confirmation message that data was stored"},
    explicit_keywords=["store", "save", "persist", "remember", "session", "memory"],
    arg_extractors={
        "session_id": _SESSION_ID_EXTRACTOR,
        "data_to_store": _QUOTED_EXTRACTOR,
    },
)


# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 14: classify_image_modality
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="classify_image_modality",
    description="Auto-detect the imaging modality and body region from the image",
    parameters={
        "patient_image": "Path to the medical image file",
    },
    required=["patient_image"],
    example={"patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg"},
    docstring="Uses MedGemma to classify imaging modality (xray, ct, mri) and body region (chest, head, abdomen) when not explicitly provided by the clinician. Returns JSON with detected image_type and body_part.",
    tool_type=ToolType.COMPUTATION,
    func=_classify_image_modality,
    arg_sources={"patient_image": "patient_image_input"},
    output_keys={"image_classification": "JSON with detected image_type and body_part"},
    explicit_keywords=["classify", "detect", "modality", "type", "identify", "what kind"],
    arg_extractors={
        "patient_image": _QUOTED_EXTRACTOR,
    },
)

# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL 15: synthesize_clinical_narrative
# ═══════════════════════════════════════════════════════════════════════════════

radiology_agent_registry.register(
    name="synthesize_clinical_narrative",
    description="Convert structured JSON findings into a concise radiology report narrative with FINDINGS and IMPRESSION sections",
    parameters={
        "verified_analysis": "Validated analysis JSON from verify_few_shot_image_analysis",
        "patient_image": "Path to the patient image file",
        "body_part": "Body region: chest|head|abdomen|brain|other",
    },
    required=["verified_analysis", "patient_image", "body_part"],
    example={
        "verified_analysis": '{"cardiomegaly": "yes", "pleural_effusion": "yes", "pneumothorax": "no"}',
        "patient_image": "/kaggle/input/chexpert/valid/patient001/view1_frontal.jpg",
        "body_part": "chest",
    },
    docstring="Transforms structured JSON analysis into attending-radiologist-style dictated prose. Produces FINDINGS (systematic, anatomic) and IMPRESSION (numbered, prioritized) sections. Constrained to 80-150 words to prevent repetition.",
    tool_type=ToolType.COMPUTATION,
    func=_synthesize_clinical_narrative,
    arg_sources={
        "verified_analysis": "verified_analysis",
        "patient_image": "patient_image_input",
        "body_part": "body_part_input",
    },
    output_keys={"clinical_narrative": "Radiologist-style FINDINGS + IMPRESSION prose"},
    explicit_keywords=["narrative", "report", "prose", "dictate", "findings", "impression", "write", "summarize"],
    arg_extractors={
        "verified_analysis": _QUOTED_EXTRACTOR,
        "patient_image": _QUOTED_EXTRACTOR,
        "body_part": _BODY_PART_EXTRACTOR,
    },
)

print(f"✓ {len(radiology_agent_registry)} tools registered (with body_part_input for Tools 2+3)")

✓ 15 tools registered (with body_part_input for Tools 2+3)


In [19]:
# ═══════════════════════════════════════════════════════════════════════════════
#  DAG VALIDATION (updated with body_part_input)
# ═══════════════════════════════════════════════════════════════════════════════
engine = DataFlowEngine(radiology_agent_registry, verbose=True)

# Validate key paths — body_part_input is now required for Tools 2+3
test_paths = {
    "verified_analysis": {"patient_image_input", "image_type_input", "body_part_input"},
    "localized_image": {"patient_image_input", "image_type_input", "body_part_input"},
    "soap_report": {"patient_image_input", "image_type_input", "body_part_input", "patient_id_input"},
    "generated_pdf_path": {"patient_image_input", "image_type_input", "body_part_input", "patient_id_input"},
    "qa_response": {"session_id_input", "followup_question_input"},
    "longitudinal_review": {"patient_image_input", "image_type_input", "body_part_input", "patient_id_input"},
}

print("\nDAG Path Validation:")
for target, sources in test_paths.items():
    plan = engine.resolve_execution_plan(sources, target)
    status = "✓" if plan else "✗"
    print(f"  {status} {target}: {plan}")

print(f"\n{engine.visualize_graph()}")

[DataFlowEngine] 15 producer keys, 15 tools registered

DAG Path Validation:
  ✓ verified_analysis: ['retrieve_similar_images', 'few_shot_image_analysis', 'verify_few_shot_image_analysis']
  ✓ localized_image: ['retrieve_similar_images', 'few_shot_image_analysis', 'verify_few_shot_image_analysis', 'localize_abnormalities']
  ✓ soap_report: ['retrieve_ehr', 'retrieve_similar_images', 'few_shot_image_analysis', 'verify_few_shot_image_analysis', 'generate_soap_report']
  ✓ generated_pdf_path: ['retrieve_ehr', 'retrieve_similar_images', 'few_shot_image_analysis', 'verify_few_shot_image_analysis', 'generate_soap_report', 'localize_abnormalities', 'build_pdf_soap_report']
  ✓ qa_response: ['retrieve_session_memory', 'qa_followup']
  ✓ longitudinal_review: ['retrieve_patient_previous_images', 'retrieve_similar_images', 'few_shot_image_analysis', 'verify_few_shot_image_analysis', 'run_longitudinal_review']


DataFlow Graph (tool -> inputs / outputs):
  retrieve_similar_images
    IN : ['patien

In [20]:
RADIOLOGY_SYSTEM_PROMPT = (
    "You are an expert Co-Radiologist AI assistant. Provide structured, "
    "clinically precise responses. Always state confidence levels. "
    "Flag critical or life-threatening findings prominently. "
    "Use standard radiology terminology."
)

# 14 examples — one per output key, concise for MedGemma-4B
RADIOLOGY_GOAL_EXAMPLES = (
    "EXAMPLES:\n"
    "Query: Find similar images -> knn_images\n"
    "Query: Analyze this chest X-ray -> few_shot_analysis\n"
    "Query: Verify the analysis -> verified_analysis\n"
    "Query: Dictate findings and impression -> clinical_narrative\n"
    "Query: Localize the abnormalities -> localized_image\n"
    "Query: Get prior imaging studies -> previous_images\n"
    "Query: Compare with previous X-ray -> longitudinal_review\n"
    "Query: Revise the report -> updated_report\n"
    "Query: Pull up patient records -> medical_records\n"
    "Query: Generate SOAP note -> soap_report\n"
    "Query: Export PDF report -> generated_pdf_path\n"
    "Query: Load previous session -> session_context\n"
    "Query: Is the effusion bilateral? -> qa_response\n"
    "Query: Save this analysis -> session_save_confirmation\n"
    "Query: What type of scan is this? -> image_classification\n"
)
print("✓ Goal examples: 14 lines, 1 per output key")

✓ Goal examples: 14 lines, 1 per output key


### **Agent Memory & Mock EHR Database**

In [21]:
from nanoathens import SessionStore 

session_store = SessionStore() 

In [22]:
import glob as _glob

# ── Discover real CheXpert validation images for prior imaging ────────────────
_valid_images = sorted(_glob.glob(
    os.path.join(KAGGLE_ROOT, "valid", "patient*", "study*", "view*_frontal.jpg")
))
if not _valid_images:
    # Fallback: try without study subdirectory
    _valid_images = sorted(_glob.glob(
        os.path.join(KAGGLE_ROOT, "valid", "patient*", "*frontal*.jpg")
    ))
print(f"Found {len(_valid_images)} validation frontal images for prior imaging")


Found 202 validation frontal images for prior imaging


In [23]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 11: MOCK INFRASTRUCTURE
#  Simulated EHR database, and image processing
#  Replace these with real implementations in production.
# ══════════════════════════════════════════════════════════════════════════════

# ── Mock Patient EHR Database ─────────────────────────────────────────────────

RADIOLOGY_EHR_DB: Dict[str, Dict] = {
    "RAD001": {
        "name": "John Smith", "age": 62, "sex": "Male",
        "complaints": "Persistent cough for 3 weeks, mild dyspnea on exertion",
        "clinical_notes": (
            "Former smoker (30 pack-years, quit 5y ago). "
            "HTN controlled on lisinopril. No hemoptysis. "
            "SpO2 96% on room air."
        ),
        "prescriptions": "Lisinopril 10mg daily, Albuterol PRN",
        "allergies": ["Penicillin"],
        "prior_imaging": [
            {
                "date": "2025-06-15", "type": "xray",
                "path": _valid_images[0] if len(_valid_images) > 0 else "prior_cxr_001.png",
                "report": "Frontal CXR: Mild cardiomegaly. Small left pleural effusion. No focal consolidation."
            },
            {
                "date": "2025-01-10", "type": "xray",
                "path": _valid_images[1] if len(_valid_images) > 1 else "prior_cxr_002.png",
                "report": "PA CXR: Heart size upper limits normal. Clear lungs bilaterally. No pleural effusion."
            },
        ],
    },

    "RAD002": {
        "name": "Maria Garcia", "age": 45, "sex": "Female",
        "complaints": "Right upper quadrant pain, nausea",
        "clinical_notes": (
            "BMI 32. Pain worse after fatty meals. Murphy sign positive. "
            "WBC 11.2. Lipase normal."
        ),
        "prescriptions": "Omeprazole 20mg daily",
        "allergies": [],
        "prior_imaging": [
            {
                "date": "2025-08-20", "type": "ct",
                "path": _valid_images[2] if len(_valid_images) > 2 else "prior_ct_abd_001.dcm",
                "report": "CT abdomen: Gallbladder wall thickening with pericholecystic fluid. Multiple gallstones. No CBD dilation."
            },
            {
                "date": "2025-03-12", "type": "xray",
                "path": _valid_images[3] if len(_valid_images) > 3 else "prior_cxr_003.png",
                "report": "Pre-op CXR: No acute cardiopulmonary process. Heart size normal."
            },
        ],
    },

    "RAD003": {
        "name": "David Chen", "age": 78, "sex": "Male",
        "complaints": "Sudden onset confusion, right-sided weakness",
        "clinical_notes": (
            "Atrial fibrillation on warfarin. INR 1.8 (subtherapeutic). "
            "NIHSS 12. Time last seen well: 2 hours ago."
        ),
        "prescriptions": "Warfarin 5mg daily, Metoprolol 50mg BID",
        "allergies": ["Iodine contrast"],
        "prior_imaging": [
            {
                "date": "2025-09-01", "type": "xray",
                "path": _valid_images[4] if len(_valid_images) > 4 else "prior_cxr_004.png",
                "report": "CXR: Cardiomegaly with signs of early pulmonary vascular congestion. No effusion."
            },
        ],
    },
}

print(f"✓ EHR DB: {len(RADIOLOGY_EHR_DB)} patients with real CheXpert prior imaging paths")
for pid, pt in RADIOLOGY_EHR_DB.items():
    n = len(pt.get("prior_imaging", []))
    paths_ok = all(os.path.exists(p["path"]) for p in pt.get("prior_imaging", []))
    print(f"  {pid} ({pt['name']}): {n} priors | paths valid: {paths_ok}")


✓ EHR DB: 3 patients with real CheXpert prior imaging paths
  RAD001 (John Smith): 2 priors | paths valid: True
  RAD002 (Maria Garcia): 2 priors | paths valid: True
  RAD003 (David Chen): 1 priors | paths valid: True


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
#  BOUNDING BOX DRAWING (Google Reference Implementation) + BM25 UTILITY
#  INSERT AS NEW CELL after EHR DB, before RadiologySessionAgent
# ══════════════════════════════════════════════════════════════════════════════

from PIL import ImageDraw, ImageFont
import skimage.util
import skimage.color
import math
from collections import Counter

# ══════════════════════════════════════════════════════════════════════════════
#  GOOGLE REFERENCE: Pad-to-Square + Scale-to-512 Bounding Box Drawing
# ══════════════════════════════════════════════════════════════════════════════

_BBOX_COLORS = [
    "red", "orange", "yellow", "lime",
    "cyan", "magenta", "violet", "deepskyblue",
]


def pad_image_to_square(image_array):
    """Pad image to square — Google MedGemma reference."""
    image_array = skimage.util.img_as_ubyte(image_array)
    if len(image_array.shape) < 3:
        image_array = skimage.color.gray2rgb(image_array)
    if image_array.shape[2] == 4:
        image_array = skimage.color.rgba2rgb(image_array)
        image_array = skimage.util.img_as_ubyte(image_array)

    h, w = image_array.shape[:2]
    max_dim = max(h, w)
    if h < w:
        dh = w - h
        image_array = np.pad(
            image_array, ((dh // 2, dh - dh // 2), (0, 0), (0, 0))
        )
    if w < h:
        dw = h - w
        image_array = np.pad(
            image_array, ((0, 0), (dw // 2, dw - dw // 2), (0, 0))
        )
    return image_array


def draw_bounding_boxes(
    image_path: str,
    bounding_boxes: list,
    output_path: str = None,
) -> str:
    """Draw bounding boxes using Google's MedGemma reference approach.

    1. Pad image to square
    2. Scale to 512px
    3. Draw boxes with [y0, x0, y1, x1] normalized to [0, 1000]
    4. Save annotated image

    Returns:
        Path to saved annotated image (in /kaggle/working/)
    """
    if not os.path.exists(image_path):
        return image_path

    # ── Load and preprocess (Google reference) ──
    img = Image.open(image_path).convert("RGB")
    image_array = np.array(img)
    image_array = (pad_image_to_square(image_array / 255.0) * 255).astype(np.uint8)
    img = Image.fromarray(image_array)

    # Scale to 512px width (preserving aspect — square so height = width)
    image_width, image_height = img.size
    new_width = 512
    new_height = int(image_height * (new_width / image_width))
    scaled_image = img.resize((new_width, new_height))

    draw = ImageDraw.Draw(scaled_image)

    # Try to load a readable font
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 14)
    except (OSError, IOError):
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 14)
        except (OSError, IOError):
            font = ImageFont.load_default()

    for i, item in enumerate(bounding_boxes):
        box_2d = item.get("box_2d")
        label = item.get("label", f"Finding {i+1}")
        color = _BBOX_COLORS[i % len(_BBOX_COLORS)]

        if not box_2d or not isinstance(box_2d, (list, tuple)) or len(box_2d) != 4:
            continue

        # Coordinates are [y0, x0, y1, x1] normalized to [0, 1000]
        y0, x0, y1, x1 = box_2d

        # Convert from normalized to pixel coordinates on scaled image
        unnormalized_x0 = x0 / 1000 * new_width
        unnormalized_y0 = y0 / 1000 * new_height
        unnormalized_x1 = x1 / 1000 * new_width
        unnormalized_y1 = y1 / 1000 * new_height

        # Draw rectangle
        draw.rectangle(
            [(unnormalized_x0, unnormalized_y0), (unnormalized_x1, unnormalized_y1)],
            outline=color,
            width=3,
        )

        # Label background + text
        if label:
            text_bbox = draw.textbbox((0, 0), label, font=font)
            text_w = text_bbox[2] - text_bbox[0]
            text_h = text_bbox[3] - text_bbox[1]
            label_y = max(unnormalized_y0 - text_h - 4, 0)
            draw.rectangle(
                [unnormalized_x0, label_y, unnormalized_x0 + text_w + 6, label_y + text_h + 2],
                fill=color,
            )
            draw.text(
                (unnormalized_x0 + 3, label_y),
                label,
                fill="white",
                font=font,
            )

    # Save to /kaggle/working/ (always accessible by Gradio)
    if output_path is None:
        stem = Path(image_path).stem
        output_path = f"/kaggle/working/localized_{stem}_{uuid.uuid4().hex[:6]}.png"

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    scaled_image.save(output_path)
    return output_path


# ══════════════════════════════════════════════════════════════════════════════
#  BM25 SEARCH UTILITY
#  Lightweight implementation — no external dependencies
#  Used by follow-up chat to find relevant context chunks
# ══════════════════════════════════════════════════════════════════════════════

def _tokenize(text: str) -> list:
    """Simple whitespace + punctuation tokenizer."""
    return re.findall(r'\b\w+\b', text.lower())


def bm25_search(query: str, documents: list, k: int = 2) -> list:
    """BM25 search over text chunks. Returns top-k most relevant chunks.

    Args:
        query: User's follow-up question
        documents: List of text chunks (context sections)
        k: Number of chunks to return

    Returns:
        Top-k document strings sorted by relevance
    """
    if not documents:
        return []
    if len(documents) <= k:
        return documents

    query_terms = _tokenize(query)
    if not query_terms:
        return documents[:k]

    # Pre-compute document frequencies
    doc_term_sets = [set(_tokenize(d)) for d in documents]
    N = len(documents)
    doc_lengths = [len(_tokenize(d)) for d in documents]
    avg_dl = sum(doc_lengths) / N if N > 0 else 1

    # BM25 parameters
    k1 = 1.5
    b = 0.75

    scores = []
    for i, doc in enumerate(documents):
        doc_terms = _tokenize(doc)
        tf = Counter(doc_terms)
        dl = doc_lengths[i]
        score = 0.0

        for term in query_terms:
            # Document frequency
            df = sum(1 for s in doc_term_sets if term in s)
            if df == 0:
                continue

            # IDF
            idf = math.log((N - df + 0.5) / (df + 0.5) + 1.0)

            # TF with BM25 normalization
            f = tf.get(term, 0)
            tf_norm = (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / avg_dl))

            score += idf * tf_norm

        scores.append((score, i))

    # Sort by score descending
    scores.sort(key=lambda x: -x[0])
    return [documents[idx] for _, idx in scores[:k]]


def chunk_context(context: str, min_chunk_size: int = 50) -> list:
    """Split session context into meaningful chunks for BM25.

    Splits on double-newlines, section headers (VERIFIED ANALYSIS, etc.),
    and Q&A pairs. Filters out tiny chunks.
    """
    # Split on section dividers
    raw_chunks = re.split(
        r'\n\n+|(?=VERIFIED ANALYSIS:)|(?=CLINICAL NARRATIVE:)|(?=SOAP REPORT:)|(?=LONGITUDINAL:)|(?=Q:)',
        context
    )
    # Filter and clean
    chunks = []
    for chunk in raw_chunks:
        chunk = chunk.strip()
        if len(chunk) >= min_chunk_size:
            chunks.append(chunk)

    return chunks if chunks else [context]


print("✓ Google-reference bounding box drawing + BM25 search utility defined")

✓ Google-reference bounding box drawing + BM25 search utility defined


### **Gemma Co-Radiologist**

In [25]:
# ══════════════════════════════════════════════════════════════════════════════
#  RADIOLOGY SESSION AGENT
#  Wrapper around DeclarativeDataFlowAgent with session state management
# ══════════════════════════════════════════════════════════════════════════════

class RadiologySessionAgent:
    """High-level Co-Radiologist agent with session state persistence.

    Wraps the DDA to provide:
      1. Automatic session creation and tracking
      2. State retention across multiple run() calls
      3. Follow-up query detection and routing
      4. Auto-save of results to session store after each execution

    Usage:
        agent = RadiologySessionAgent(reasoning_caller=run_medgemma)
        r1 = await agent.run("Analyze this chest X-ray for patient RAD001",
                              image="uploads/cxr_001.png", image_type="xray")
        # Follow-up (same session):
        r2 = await agent.run("What about the pleural effusion?")
        # New session:
        r3 = await agent.run("Review CT head for patient RAD003",
                              image="ct_head.dcm", image_type="ct",
                              new_session=True)
    """

    def __init__(
        self,
        reasoning_caller: Callable = None,
        verbose: bool = True,
    ):
        self.reasoning_llm = reasoning_caller or run_medgemma
        self.verbose = verbose
        self.session_id: Optional[str] = None

        # Create the DDA with radiology configuration
        self.dda = DeclarativeDataFlowAgent(
            registry=radiology_agent_registry,
            reasoning_caller=self.reasoning_llm,
            verbose=verbose,
            system_prompt=RADIOLOGY_SYSTEM_PROMPT,
            goal_few_shot_examples=RADIOLOGY_GOAL_EXAMPLES,
        )

    def _log(self, msg):
        if self.verbose:
            print(f"[RadSession] {msg}")

    def _is_followup(self, query: str) -> bool:
        """Detect if a query is a follow-up question vs. a new analysis."""
        followup_signals = [
            "what about", "tell me more", "explain", "is there",
            "can you clarify", "regarding", "the ", "that ",
            "you mentioned", "earlier", "previous", "above",
            "how about", "why", "could you",
        ]
        ql = query.lower().strip()

        # Short queries are likely follow-ups
        if len(ql.split()) <= 8:
            for signal in followup_signals:
                if signal in ql:
                    return True

        # If no image reference and we have a session, likely follow-up
        image_signals = ["image", "scan", "x-ray", "xray", "ct", "mri",
                          "analyze", "review", "read"]
        has_image_ref = any(s in ql for s in image_signals)
        return not has_image_ref and self.session_id is not None

    async def run(
        self,
        query: str,
        image: str = None,
        image_type: str = None,
        body_part: str = None,
        patient_id: str = None,
        target_key: str = None,
        new_session: bool = False,
    ) -> Dict:
        """Execute a radiology query with session state management."""
        # Session management
        if new_session or self.session_id is None:
            self.session_id = SESSION_STORE.create_session()
            self._log(f"New session: {self.session_id}")

        # Detect follow-up vs. new analysis
        is_followup = self._is_followup(query) and image is None

        if is_followup:
            self._log("Detected follow-up question — routing through QA tool")
            session_summary = SESSION_STORE.get_session_summary(self.session_id)
            if target_key is None:
                target_key = "qa_response"
            augmented_query = (
                f"{query}\n\n"
                f"[PRIOR SESSION DATA]\n{session_summary}"
            )
        else:
            # New analysis: build augmented query with provided metadata
            parts = [query]
            if image:
                parts.append(f"Patient image: '{image}'")
                SESSION_STORE.save_context(
                    self.session_id, "patient_image_input", image
                )
            if image_type:
                parts.append(f"Image type: {image_type}")
                SESSION_STORE.save_context(
                    self.session_id, "image_type_input", image_type
                )
            if body_part:
                parts.append(f"Body part: {body_part}")
                SESSION_STORE.save_context(
                    self.session_id, "body_part_input", body_part
                )
            if patient_id:
                parts.append(f"Patient ID: {patient_id}")
                SESSION_STORE.save_context(
                    self.session_id, "patient_id_input", patient_id
                )
            augmented_query = " | ".join(parts)

        # Run DDA
        result = await self.dda.run(augmented_query, target_key=target_key)

        # Save results to session store
        SESSION_STORE.save_run_result(self.session_id, query, result)

        # Save key tool outputs to session context for future follow-ups
        for key in result.get("context_keys", []):
            SESSION_STORE.save_context(self.session_id, key, "computed")

        # Save the response itself
        SESSION_STORE.save_context(
            self.session_id,
            "last_response",
            result.get("response", "")[:1000],
        )

        # Enrich result with session info
        result["session_id"] = self.session_id
        result["is_followup"] = is_followup

        return result

    def get_session_history(self) -> Dict:
        """Get the full session history for debugging/inspection."""
        if self.session_id:
            return SESSION_STORE.get_full_session(self.session_id)
        return {}

    def reset_session(self):
        """Start a fresh session."""
        self.session_id = None


print("✓ RadiologySessionAgent defined (with body_part support)")

✓ RadiologySessionAgent defined (with body_part support)


In [27]:
# ══════════════════════════════════════════════════════════════════════════════
#  DATAFLOW GRAPH INSPECTION
# ══════════════════════════════════════════════════════════════════════════════

def inspect_radiology_graph():
    """Print the complete dataflow graph and stats."""
    engine = DataFlowEngine(radiology_agent_registry, verbose=True)
    stats = engine.get_graph_stats()
    print("\nGraph Stats:")
    for k, v in stats.items():
        print(f"  {k}: {v}")
    print(engine.visualize_graph())
    return engine

### **Agent UI Section**

In [ ]:
#!pip install -q gradio
#!pip install -q gradio==5.50.0

In [28]:
# ── Kill uvloop before anything loads it ──
import sys
sys.modules['uvloop'] = None

import asyncio
asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

In [33]:
def get_review_schema(image_type: str, body_part: str = "chest"):
    key = (image_type.lower().strip(), body_part.lower().strip())
    return REVIEW_SCHEMA_MAP.get(key, CXRReview)


def _revise_structured_analysis(original_json: str, critique: str, image_type: str, body_part: str) -> str:
    """
    Ask the LLM to update the structured findings JSON according to the clinician's critique.
    """
    schema_cls = get_review_schema(image_type, body_part)
    prompt = f"""You are an expert radiologist revising a structured analysis based on clinician feedback.

Original structured analysis (JSON):
{original_json}

Clinician's feedback:
{critique}

Please produce an updated JSON object that incorporates the feedback. Follow the same schema exactly: {schema_cls.model_json_schema()}

Only output the JSON, no other text.
"""
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    response = run_medgemma(messages, max_new_tokens=512, temperature=0.1)
    validated = extract_and_validate_json(response, schema_cls, run_medgemma)
    return json.dumps(validated)

In [34]:
# ══════════════════════════════════════════════════════════════════════════════
#  Gradio GUI — Gemma Co-Radiologist (Competition UI v3)
#  REPLACES CELL 35 ENTIRELY
#  - Single "Run Deep Analysis" trigger
#  - Progressive step notifications via generator yields
#  - Gallery on right panel under tabs
#  - 1.5x font scaling for video demo
#  - Fast localization (max 2 findings to avoid P100 timeout)
# ══════════════════════════════════════════════════════════════════════════════

!pip install -q reportlab 2>/dev/null || true

import gradio as gr
import shutil

# ══════════════════════════════════════════════════════════════════════════════
#  SESSION AGENT
# ══════════════════════════════════════════════════════════════════════════════

session_agent = RadiologySessionAgent(
    reasoning_caller=run_medgemma,
    verbose=False,
)


# ══════════════════════════════════════════════════════════════════════════════
#  FAST LOCALIZATION WRAPPER
#  Caps to MAX 2 findings to prevent MedGemma from running 5+ inference calls
#  on a P100 (each call ~60s → 5 findings = 5 min stuck)
# ══════════════════════════════════════════════════════════════════════════════

MAX_LOCALIZE_FINDINGS = 2

def _fast_localize(verified_analysis: str, patient_image: str, narrative: str = "") -> str:
    """Localize only findings that appear in the clinical narrative.

    This ensures bounding boxes match what the clinician reads in the report,
    not random JSON keys from the structured analysis.

    Falls back to top-2 positive JSON findings if no narrative is available.
    """
    try:
        analysis = json.loads(verified_analysis) if isinstance(verified_analysis, str) else verified_analysis
    except (json.JSONDecodeError, TypeError):
        return _localize_abnormalities(verified_analysis, patient_image)

    if not isinstance(analysis, dict):
        return _localize_abnormalities(verified_analysis, patient_image)

    skip = {"_meta", "verification_confidence", "verification_notes",
            "view", "section", "other_abnormal_features", "other_findings"}

    # Collect all positive findings from JSON
    yes_keys, unclear_keys = [], []
    for k, v in analysis.items():
        if k in skip or k.startswith("_"):
            continue
        val = str(v).lower()
        if val == "yes":
            yes_keys.append(k)
        elif val == "unclear":
            unclear_keys.append(k)

    all_positive = yes_keys + unclear_keys

    if narrative:
        # Filter: only keep findings that are actually mentioned in the narrative
        narrative_lower = narrative.lower()
        mentioned = []
        for k in all_positive:
            # Convert key to readable form for matching
            readable = k.replace("_", " ").lower()
            # Also check common synonyms
            synonyms = {
                "lung_opacity": ["opacity", "opacit"],
                "pleural_effusion": ["effusion"],
                "cardiomegaly": ["cardiomegal", "enlarged heart", "cardiac silhouette"],
                "edema": ["edema", "congestion"],
                "consolidation": ["consolidat"],
                "atelectasis": ["atelectas"],
                "pneumothorax": ["pneumothorax"],
                "enlarged_cardiomediastinum": ["cardiomediastin", "mediastin"],
                "lung_lesion": ["lesion", "mass", "nodule"],
                "pneumonia": ["pneumonia"],
                "fracture": ["fracture"],
                "support_devices": ["device", "line", "tube", "catheter"],
            }
            terms_to_check = [readable] + synonyms.get(k, [])
            if any(term in narrative_lower for term in terms_to_check):
                mentioned.append(k)

        # Use narrative-matched findings (capped at MAX)
        if mentioned:
            keep_positive = mentioned[:MAX_LOCALIZE_FINDINGS]
        else:
            # Fallback if narrative matching fails
            keep_positive = all_positive[:MAX_LOCALIZE_FINDINGS]
    else:
        keep_positive = all_positive[:MAX_LOCALIZE_FINDINGS]

    # Build trimmed analysis with only selected positive findings + all negatives
    trimmed = {}
    for k in keep_positive:
        trimmed[k] = analysis[k]
    for k, v in analysis.items():
        if k in skip or k.startswith("_"):
            if k in analysis:
                trimmed[k] = analysis[k]
            continue
        if str(v).lower() not in ("yes", "unclear", "true"):
            trimmed[k] = v

    return _localize_abnormalities(json.dumps(trimmed), patient_image)


# ══════════════════════════════════════════════════════════════════════════════
#  STATUS HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _status_html(title, message, kind="info"):
    colors = {
        "info": ("#0ea5e9", "#0c2d4a"),
        "running": ("#0ea5e9", "#0c2d4a"),
        "error": ("#ef4444", "#2d0c0c"),
        "success": ("#22c55e", "#0c2d1a"),
    }
    fg, bg = colors.get(kind, colors["info"])
    return f"""
    <div style="padding:12px 16px;background:{bg};border:1px solid {fg}33;border-radius:8px;">
        <span style="color:{fg};font-weight:600;font-size:1.05em;">{title}</span>
        <span style="color:#94a3b8;margin-left:8px;font-size:0.95em;">{message}</span>
    </div>"""


def _progress_html(completed_steps, current_step, t0, total=7):
    pct = int((len(completed_steps) / total) * 100)
    lines = []
    for s in completed_steps:
        lines.append(f'<div style="color:#22c55e;font-size:0.95em;padding:1px 0;">✓ {s}</div>')
    if current_step:
        lines.append(f'<div style="color:#0ea5e9;font-size:0.95em;padding:1px 0;">⟳ {current_step}...</div>')
    steps_html = "".join(lines)
    elapsed = round(time.time() - t0, 1)
    return f"""
    <div style="padding:10px 0;">
        <div style="display:flex;align-items:center;gap:12px;margin-bottom:10px;">
            <div style="flex:1;height:7px;background:#1a2744;border-radius:4px;overflow:hidden;">
                <div style="width:{pct}%;height:100%;background:linear-gradient(90deg,#0ea5e9,#06b6d4);border-radius:4px;transition:width 0.5s ease;"></div>
            </div>
            <span style="color:#94a3b8;font-size:0.9em;white-space:nowrap;">{pct}% · {elapsed}s</span>
        </div>
        {steps_html}
    </div>"""


def _final_html(completed_steps, elapsed):
    lines = "".join(f'<div style="color:#22c55e;font-size:0.95em;padding:1px 0;">✓ {s}</div>' for s in completed_steps)
    return f"""
    <div style="padding:10px 0;">
        <div style="display:flex;align-items:center;gap:12px;margin-bottom:10px;">
            <div style="flex:1;height:7px;background:#1a2744;border-radius:4px;overflow:hidden;">
                <div style="width:100%;height:100%;background:linear-gradient(90deg,#22c55e,#16a34a);border-radius:4px;"></div>
            </div>
            <span style="color:#22c55e;font-size:0.9em;font-weight:600;">✓ Complete · {elapsed}s</span>
        </div>
        {lines}
    </div>"""


# ══════════════════════════════════════════════════════════════════════════════
#  PROGRESSIVE DEEP ANALYSIS GENERATOR
#
#  Output order (16 elements, matches _analysis_outputs):
#    0  status_box            (HTML)
#    1  narrative_output      (Markdown)
#    2  localized_image       (Image)
#    3  localization_info     (Markdown)
#    4  soap_output           (Textbox)
#    5  prior_image_display   (Image)
#    6  current_image_display (Image)
#    7  longitudinal_output   (Markdown)
#    8  pdf_download          (File)
#    9  pdf_status            (Markdown)
#    10 similar_gallery       (Gallery)
#    11 findings_state        (State)
#    12 context_state         (State)
#    13 img_path_state        (State)
#    14 soap_state            (State)
#    15 loc_state             (State)
# ══════════════════════════════════════════════════════════════════════════════

S = gr.skip()


def run_deep_analysis(image, image_type, body_part, patient_id):
    """Single-trigger deep analysis. Generator yields progressive updates."""
    t0 = time.time()

    img_path = image if isinstance(image, str) else (image.name if image else "")
    if not img_path or not os.path.exists(img_path):
        yield (
            _status_html("❌ Error", "No image uploaded.", "error"),
            S, S, S, S, S, S, S, S, S, S,
            S, S, S, S, S,
        )
        return

    pid = patient_id if patient_id and patient_id != "None" else None
    steps = []

    # ── STEP 1: Similar Images ───────────────────────────────────────────────
    yield (
        _progress_html(steps, "Retrieving similar cases from database", t0),
        S, S, S, S, S, S, S, S, S, S,
        S, S, S, S, S,
    )

    try:
        knn_raw = _retrieve_similar_images(img_path, image_type)
        knn_data = json.loads(knn_raw) if isinstance(knn_raw, str) else knn_raw
        similar = knn_data.get("similar_images", []) if isinstance(knn_data, dict) else []
        gallery_items = []
        for item in similar[:5]:
            rel_path = item.get("path", "")
            full_path = os.path.join(KAGGLE_ROOT, rel_path)
            score = item.get("score", 0)
            desc = item.get("description", "")[:80]
            if os.path.exists(full_path):
                gallery_items.append((full_path, f"Score: {score:.3f} | {desc}"))
    except Exception:
        knn_raw = "{}"
        gallery_items = []

    steps.append("Similar cases retrieved")
    yield (
        _progress_html(steps, "Analyzing image with few-shot context", t0),
        S, S, S, S, S, S, S, S, S,
        gallery_items,  # similar_gallery (#10)
        S, S, S, S, S,
    )

    # ── STEP 2: Few-Shot Analysis ────────────────────────────────────────────
    try:
        analysis_raw = _few_shot_image_analysis(knn_raw, img_path, body_part)
    except Exception:
        analysis_raw = "{}"

    steps.append("Preliminary analysis complete")
    yield (
        _progress_html(steps, "Verifying analysis (senior radiologist pass)", t0),
        S, S, S, S, S, S, S, S, S, S,
        S, S, S, S, S,
    )

    # ── STEP 3: Verify Analysis ──────────────────────────────────────────────
    try:
        verified_raw = _verify_few_shot_image_analysis(analysis_raw, img_path, body_part)
    except Exception:
        verified_raw = analysis_raw

    steps.append("Analysis verified")
    yield (
        _progress_html(steps, "Synthesizing clinical narrative", t0),
        S, S, S, S, S, S, S, S, S, S,
        S, S, S, S, S,
    )

    # ── STEP 4: Clinical Narrative ───────────────────────────────────────────
    try:
        narrative = _synthesize_clinical_narrative(verified_raw, img_path, body_part)
    except Exception as e:
        narrative = f"Narrative generation failed: {e}"

    steps.append("Clinical narrative synthesized")
    yield (
        _progress_html(steps, "Localizing abnormalities (max 2 findings)", t0),
        narrative,  # narrative_output (#1)
        S, S, S, S, S, S, S, S, S,
        S, S, S, S, S,
    )

    # ── STEP 5: Localize (FAST — max 2 findings) ────────────────────────────
    annotated_path = None
    loc_summary = "No localizable abnormalities detected."
    loc_raw = "{}"
    try:
        loc_result = _fast_localize(verified_raw, img_path, narrative)
        loc_raw = loc_result
        loc_data = json.loads(loc_result) if isinstance(loc_result, str) else loc_result
        bboxes = loc_data.get("bounding_boxes", [])
        findings = loc_data.get("findings", [])

        if bboxes and any(b.get("box_2d") for b in bboxes):
            annotated_path = draw_bounding_boxes(img_path, bboxes)
            parts = [f"**{len(bboxes)}** regions localized:"]
            for b in bboxes:
                label = b.get("label", "Unknown")
                coords = b.get("box_2d", "N/A")
                parts.append(f"• **{label}**: {coords}")
            loc_summary = "\n".join(parts)
        elif findings:
            loc_summary = (
                f"Findings detected ({', '.join(findings)}) but bounding boxes "
                "could not be resolved. Manual review recommended."
            )
            if os.path.exists(img_path):
                annotated_path = img_path
    except Exception as e:
        loc_summary = f"Localization error: {e}"

    steps.append("Abnormalities localized")
    yield (
        _progress_html(steps, "Generating SOAP report", t0),
        S,
        annotated_path,  # localized_image (#2)
        loc_summary,     # localization_info (#3)
        S, S, S, S, S, S, S,
        S, S, S, S, S,
    )

    # ── STEP 6: SOAP Report ──────────────────────────────────────────────────
    soap_text = ""
    try:
        ehr_pid = pid or "RAD001"
        ehr = _retrieve_ehr(ehr_pid)
        soap_text = _generate_soap_report(ehr, verified_raw)
    except Exception as e:
        soap_text = f"SOAP generation failed: {e}"

    steps.append("SOAP report generated")

    has_priors = False
    if pid:
        pt = RADIOLOGY_EHR_DB.get(pid, {})
        has_priors = len(pt.get("prior_imaging", [])) > 0

    next_label = "Running longitudinal review" if has_priors else "Building PDF report"
    yield (
        _progress_html(steps, next_label, t0),
        S, S, S,
        soap_text,  # soap_output (#4)
        S, S, S, S, S, S,
        S, S, S, S, S,
    )

    # ── STEP 6.5: Longitudinal Review ────────────────────────────────────────
    prior_img = None
    current_img = None
    longitudinal_text = "No prior studies available for comparison."

    # Copy current image to /kaggle/working/ so Gradio can always serve it
    if os.path.exists(img_path):
        current_copy = f"/kaggle/working/current_{uuid.uuid4().hex[:6]}.png"
        shutil.copy2(img_path, current_copy)
        current_img = current_copy

    if has_priors and pid:
        try:
            priors_json = _retrieve_patient_previous_images(pid, image_type)
            priors = json.loads(priors_json) if isinstance(priors_json, str) else priors_json
            prior_list = priors.get("previous_images", [])

            if prior_list:
                prior_path = prior_list[0].get("path", "")
                if os.path.exists(prior_path):
                    # Copy prior image to /kaggle/working/ for Gradio access
                    prior_copy = f"/kaggle/working/prior_{uuid.uuid4().hex[:6]}.png"
                    shutil.copy2(prior_path, prior_copy)
                    prior_img = prior_copy

                review = _run_longitudinal_review(img_path, priors_json, verified_raw)
                prior_info = "\n".join(
                    f"• **{p.get('date', '?')}** [{p.get('type', '?').upper()}]: {p.get('report', 'No report')}"
                    for p in prior_list
                )
                longitudinal_text = f"**Prior Studies:**\n{prior_info}\n\n---\n\n**Interval Change Assessment:**\n{review}"
                steps.append("Longitudinal review complete")
        except Exception as e:
            longitudinal_text = f"Longitudinal review error: {e}"

    yield (
        _progress_html(steps, "Building PDF report", t0),
        S, S, S, S,
        prior_img,          # prior_image_display (#5)
        current_img,        # current_image_display (#6)
        longitudinal_text,  # longitudinal_output (#7)
        S, S, S,
        S, S, S, S, S,
    )

    # ── STEP 7: PDF Report ───────────────────────────────────────────────────
    pdf_path = None
    pdf_msg = ""
    try:
        pdf_result = _build_pdf_soap_report(soap_text, loc_raw, img_path)
        pdf_data = json.loads(pdf_result) if isinstance(pdf_result, str) else pdf_result
        pdf_file = pdf_data.get("generated_pdf_path", "")
        if os.path.exists(pdf_file):
            pdf_path = pdf_file
            pdf_msg = f"✓ Report ready for download: `{pdf_file}`"
        else:
            pdf_msg = "PDF path generated but file not found."
    except Exception as e:
        pdf_msg = f"PDF generation error: {e}"

    steps.append("PDF report built")
    elapsed = round(time.time() - t0, 1)

    context = (
        f"VERIFIED ANALYSIS:\n{verified_raw}\n\n"
        f"CLINICAL NARRATIVE:\n{narrative}\n\n"
        f"SOAP REPORT:\n{soap_text}\n\n"
        f"LONGITUDINAL:\n{longitudinal_text}"
    )

    # ── FINAL YIELD ──────────────────────────────────────────────────────────
    yield (
        _final_html(steps, elapsed),  # status_box (#0)
        S,                            # narrative (#1 - already set)
        S,                            # localized_image (#2 - already set)
        S,                            # localization_info (#3 - already set)
        S,                            # soap (#4 - already set)
        S,                            # prior_image (#5 - already set)
        S,                            # current_image (#6 - already set)
        S,                            # longitudinal (#7 - already set)
        pdf_path,                     # pdf_download (#8)
        pdf_msg,                      # pdf_status (#9)
        S,                            # gallery (#10 - already set)
        verified_raw,                 # findings_state (#11)
        context,                      # context_state (#12)
        img_path,                     # img_path_state (#13)
        soap_text,                    # soap_state (#14)
        loc_raw,                      # loc_state (#15)
    )


# ══════════════════════════════════════════════════════════════════════════════
#  CHAT HANDLER
# ══════════════════════════════════════════════════════════════════════════════

def chat_handler(message, history, context_state, chat_mode,
                 findings_json, img_path, body_part, patient_id,
                 soap_text_current):
    """
    Output order (10 elements, matches _chat_outputs):
      0 chatbot, 1 chat_input, 2 context_state,
      3 narrative_output, 4 localized_image, 5 localization_info,
      6 soap_output, 7 pdf_download, 8 pdf_status, 9 findings_state
    """
    if not message or not message.strip():
        return history or [], "", context_state, S, S, S, S, S, S, S

    if not context_state:
        history = history or []
        history.append({"role": "user", "content": message})
        history.append({
            "role": "assistant",
            "content": "No analysis context yet. Please run **🔬 Run Deep Analysis** first."
        })
        return history, "", context_state, S, S, S, S, S, S, S

    history = history or []
    history.append({"role": "user", "content": message})

    new_narrative = S
    new_loc_img = S
    new_loc_info = S
    new_soap = S
    new_pdf = S
    new_pdf_msg = S
    new_findings = S

    if chat_mode == "Revise":
      try:
        # Step 1: Revise the structured JSON using the critique
        # Detect image_type from the findings JSON (or fallback)
        try:
            orig = json.loads(findings_json) if findings_json else {}
        except:
            orig = {}
        # Determine image_type by looking at keys (same as in _verify_few_shot_image_analysis)
        ct_head_fields = {"hemorrhage", "midline_shift", "hydrocephalus", "calvarial_fracture"}
        ct_abdo_fields = {"bowel_obstruction", "appendicitis", "hepatic_mass", "pneumoperitoneum"}
        orig_keys = set(orig.keys()) if isinstance(orig, dict) else set()
        if orig_keys & ct_head_fields:
            image_type = "ct"
            if body_part not in ("head", "brain"):
                body_part = "head"
        elif orig_keys & ct_abdo_fields:
            image_type = "ct"
            if body_part != "abdomen":
                body_part = "abdomen"
        else:
            image_type = "xray"
            if body_part != "chest":
                body_part = "chest"

        # Get the revised JSON
        revised_json = _revise_structured_analysis(findings_json, message, image_type, body_part)
        new_findings = revised_json

        # Step 2: Regenerate narrative from the updated JSON
        try:
            new_narrative = _synthesize_clinical_narrative(revised_json, img_path, body_part)
        except Exception as e:
            new_narrative = f"Narrative regeneration failed: {e}"

        # Step 3: Relocalize using the updated JSON and the new narrative
        try:
            loc_result = _fast_localize(revised_json, img_path, new_narrative if isinstance(new_narrative, str) else "")
            loc_data = json.loads(loc_result) if isinstance(loc_result, str) else loc_result
            bboxes = loc_data.get("bounding_boxes", [])
            if bboxes and any(b.get("box_2d") for b in bboxes):
                new_loc_img = draw_bounding_boxes(img_path, bboxes)
                parts = [f"**{len(bboxes)}** regions re-localized:"]
                for b in bboxes:
                    parts.append(f"• **{b.get('label', '?')}**: {b.get('box_2d', 'N/A')}")
                new_loc_info = "\n".join(parts)
            else:
                new_loc_info = "No localizable abnormalities after revision."
        except Exception as e:
            new_loc_info = f"Relocalization failed: {e}"

        # Step 4: Regenerate SOAP using the updated JSON
        try:
            ehr_pid = patient_id if patient_id and patient_id != "None" else "RAD001"
            ehr = _retrieve_ehr(ehr_pid)
            new_soap = _generate_soap_report(ehr, revised_json)
        except Exception as e:
            new_soap = soap_text_current or f"SOAP regeneration failed: {e}"

        # Step 5: Rebuild PDF with the new SOAP and localization
        try:
            _soap_for_pdf = new_soap if isinstance(new_soap, str) else (soap_text_current or "")
            _loc_for_pdf = loc_result if 'loc_result' in dir() else "{}"
            pdf_result = _build_pdf_soap_report(_soap_for_pdf, _loc_for_pdf, img_path)
            pdf_data = json.loads(pdf_result) if isinstance(pdf_result, str) else pdf_result
            pf = pdf_data.get("generated_pdf_path", "")
            if os.path.exists(pf):
                new_pdf = pf
                new_pdf_msg = f"✓ PDF regenerated with revisions: `{pf}`"
        except Exception as e:
            new_pdf_msg = f"PDF regeneration failed: {e}"

        response = (
            "**Report revised.** Your feedback has been incorporated into the analysis. "
            "The narrative, localization, SOAP report, and PDF have all been regenerated."
        )

      except Exception as e:
        response = f"⚠ Revision failed: {e}"
        new_findings = S


      
       # ============================================
    else:
        # ── Follow-up mode: BM25 retrieval + Q&A ──
        # Chunk the context and find the 2 most relevant sections
        chunks = chunk_context(context_state)
        relevant_chunks = bm25_search(message, chunks, k=2)
        focused_context = "\n\n".join(relevant_chunks)

        prompt = (
            f"You are an expert Co-Radiologist answering a follow-up question.\n\n"
            f"RELEVANT CONTEXT (retrieved via search):\n{focused_context}\n\n"
            f"CLINICIAN'S QUESTION: {message}\n\n"
            f"Answer using ONLY information from the context above. "
            f"If the information is not available, say so clearly."
        )
        messages_llm = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        response = run_medgemma(messages=messages_llm, max_new_tokens=512, temperature=0.2)

    history.append({"role": "assistant", "content": response})
    updated_ctx = context_state + f"\n\nQ: {message}\nA: {response}"

    return (
        history, "", updated_ctx,
        new_narrative, new_loc_img, new_loc_info,
        new_soap, new_pdf, new_pdf_msg, new_findings,
    )


# ══════════════════════════════════════════════════════════════════════════════
#  PATIENT INFO
# ══════════════════════════════════════════════════════════════════════════════

def get_patient_info(patient_id):
    if not patient_id or patient_id == "None":
        return "Select a patient to view their clinical records."
    pt = RADIOLOGY_EHR_DB.get(patient_id, {})
    if not pt:
        return f"Patient {patient_id} not found."
    priors = pt.get("prior_imaging", [])
    prior_lines = "\n".join(
        f"  • {p['date']} [{p['type'].upper()}] — {p['report'][:60]}..."
        for p in priors
    ) if priors else "  None on file"
    return (
        f"**{pt.get('name', 'Unknown')}** · {pt.get('age', '?')}y · {pt.get('sex', '?')}\n\n"
        f"**Chief Complaint:** {pt.get('complaints', 'N/A')}\n\n"
        f"**Clinical Notes:** {pt.get('clinical_notes', 'N/A')}\n\n"
        f"**Medications:** {pt.get('prescriptions', 'None')}\n\n"
        f"**Allergies:** {', '.join(pt.get('allergies', [])) or 'NKDA'}\n\n"
        f"**Prior Imaging ({len(priors)} studies):**\n{prior_lines}"
    )


# ══════════════════════════════════════════════════════════════════════════════
#  DAG
# ══════════════════════════════════════════════════════════════════════════════

def get_graph_info():
    engine = DataFlowEngine(radiology_agent_registry, verbose=False)
    rows = []
    for name in radiology_agent_registry.get_all_names():
        tool = radiology_agent_registry.get_tool(name)
        inputs = ", ".join(tool.arg_sources.values()) or "(leaf)"
        outputs = ", ".join(tool.output_keys.keys())
        rows.append([name, tool.tool_type.value.upper(), inputs, outputs])
    stats = engine.get_graph_stats()
    return rows, f"**{stats['total_tools']}** tools · **{stats['unique_producer_keys']}** data keys · **{len(stats['source_tools'])}** source tools"

def get_dag_paths():
    engine = DataFlowEngine(radiology_agent_registry, verbose=False)
    targets = {
        "verified_analysis": {"patient_image_input", "image_type_input", "body_part_input"},
        "clinical_narrative": {"patient_image_input", "image_type_input", "body_part_input"},
        "localized_image": {"patient_image_input", "image_type_input", "body_part_input"},
        "soap_report": {"patient_image_input", "image_type_input", "body_part_input", "patient_id_input"},
        "longitudinal_review": {"patient_image_input", "image_type_input", "body_part_input", "patient_id_input"},
        "generated_pdf_path": {"patient_image_input", "image_type_input", "body_part_input", "patient_id_input"},
        "qa_response": {"session_id_input", "followup_question_input"},
    }
    lines = []
    for target, sources in targets.items():
        plan = engine.resolve_execution_plan(sources, target)
        status = "✓" if plan else "✗"
        chain = " → ".join(plan) if plan else "NO PATH"
        lines.append(f"{status}  **{target}**\n    {chain}")
    return "\n\n".join(lines)

_patient_ids = ["None"] + list(RADIOLOGY_EHR_DB.keys())
_graph_rows, _graph_summary = get_graph_info()
_dag_paths = get_dag_paths()


# ══════════════════════════════════════════════════════════════════════════════
#  CSS — Dark theme, 1.5x fonts, gallery on right
# ══════════════════════════════════════════════════════════════════════════════

CUSTOM_CSS = """
/* ── Global: 1.5x font scaling ── */
.gradio-container {
    max-width: 1440px !important;
    margin: 0 auto !important;
    background: #070b14 !important;
    font-size: 16.5px !important;
}
.gradio-container input,
.gradio-container textarea,
.gradio-container select {
    color: #e2e8f0 !important;
    font-size: 1em !important;
}
.gradio-container label,
.gradio-container .label-wrap span {
    font-size: 1.05em !important;
}
.gradio-container .prose {
    font-size: 1em !important;
    line-height: 1.7 !important;
}
.gradio-container button {
    font-size: 1em !important;
}
.gradio-container table th,
.gradio-container table td {
    font-size: 0.95em !important;
}

/* ── Header ── */
.header-banner {
    background: linear-gradient(135deg, #0c1829 0%, #0a1628 40%, #0d1f3c 100%);
    border: 1px solid #1a2744;
    border-radius: 12px;
    padding: 28px 36px;
    margin-bottom: 16px;
    position: relative;
    overflow: hidden;
}
.header-banner::before {
    content: '';
    position: absolute;
    top: -50%; left: -50%;
    width: 200%; height: 200%;
    background: radial-gradient(ellipse at 30% 50%, rgba(14,165,233,0.06) 0%, transparent 60%),
                radial-gradient(ellipse at 70% 50%, rgba(6,182,212,0.04) 0%, transparent 60%);
    pointer-events: none;
}
.header-banner h1 {
    font-size: 2.4em !important; font-weight: 700 !important;
    color: #e2e8f0 !important; margin: 0 0 6px 0 !important;
    letter-spacing: -0.02em;
}
.header-banner p {
    color: #64748b !important; font-size: 1.1em !important; margin: 0 !important;
}
.header-banner .accent { color: #0ea5e9; }
.header-banner .tag {
    display: inline-block;
    background: rgba(14,165,233,0.12); color: #38bdf8;
    padding: 3px 14px; border-radius: 20px; font-size: 0.45em;
    border: 1px solid rgba(14,165,233,0.2);
    margin-left: 10px; vertical-align: middle;
}

/* ── Deep Analysis Button ── */
.deep-analysis-btn {
    background: linear-gradient(135deg, #0ea5e9, #0284c7) !important;
    border: 1px solid #0ea5e9 !important;
    color: white !important;
    font-size: 1.15em !important;
    padding: 16px 24px !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    transition: all 0.25s ease !important;
}
.deep-analysis-btn:hover {
    box-shadow: 0 0 24px rgba(14,165,233,0.35) !important;
    transform: translateY(-1px) !important;
}

/* ── Narrative ── */
.narrative-box {
    font-size: 1.05em !important;
    line-height: 1.75 !important;
}

/* ── Chat divider ── */
.chat-section-divider {
    border-top: 1px solid #1a2744;
    margin-top: 16px;
    padding-top: 4px;
}

/* ── EHR ── */
.ehr-panel {
    background: rgba(14,165,233,0.04) !important;
    border: 1px solid rgba(14,165,233,0.1) !important;
    border-radius: 8px !important;
}

/* ── Dropdown & input labels — bright white ── */
.gradio-container .label-wrap,
.gradio-container .label-wrap span,
.gradio-container label span,
.gradio-container .gr-dropdown label,
.gradio-container .gr-input label {
    color: #f1f5f9 !important;
}
/* ── Dropdown selected value text — bright ── */
.gradio-container select,
.gradio-container .wrap .wrap-inner,
.gradio-container input[type="text"],
.gradio-container .secondary-wrap,
.gradio-container [data-testid="dropdown"] span,
.gradio-container .dropdown-arrow {
    color: #f1f5f9 !important;
}

/* ── Tabs ── */
.tab-nav button {
    color: #64748b !important; font-weight: 600 !important;
    font-size: 1.05em !important;
    border-bottom: 2px solid transparent !important;
}
.tab-nav button.selected {
    color: #0ea5e9 !important;
    border-bottom-color: #0ea5e9 !important;
}

/* ── Scroll ── */
.scroll-output textarea {
    max-height: 420px; overflow-y: auto !important;
    font-size: 1em !important;
}

/* ── Chatbot messages ── */
.chatbot .message {
    font-size: 1em !important;
}
/* ── Chatbot text — black for video readability ── */
.chatbot .message-wrap .message,
.chatbot .bot,
.chatbot .user,
.chatbot .message-wrap,
.chatbot .message-wrap p,
.chatbot .message-wrap span,
.chatbot [data-testid="bot"],
.chatbot [data-testid="user"],
.gradio-container .chatbot .prose,
.gradio-container .chatbot .prose p,
.gradio-container .chatbot .prose strong,
.gradio-container .chatbot .markdown-text,
.gradio-container .chatbot .markdown-text p {
    color: #000000 !important;
}
/* ── File download text — black ── */
.gradio-container .file-preview,
.gradio-container .file-preview a,
.gradio-container .file-preview span,
.gradio-container [data-testid="file"] a,
.gradio-container [data-testid="file"] span {
    color: #000000 !important;
}
"""


# ══════════════════════════════════════════════════════════════════════════════
#  BUILD INTERFACE
# ══════════════════════════════════════════════════════════════════════════════

with gr.Blocks(
    title="Gemma Co-Radiologist",
    theme=gr.themes.Base(
        primary_hue="sky",
        secondary_hue="cyan",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("Plus Jakarta Sans"),
        font_mono=gr.themes.GoogleFont("JetBrains Mono"),
    ).set(
        body_background_fill="#070b14",
        body_background_fill_dark="#070b14",
        block_background_fill="#0c1221",
        block_background_fill_dark="#0c1221",
        block_border_color="#1a2744",
        block_border_color_dark="#1a2744",
        block_label_text_color="#94a3b8",
        block_label_text_color_dark="#94a3b8",
        block_title_text_color="#e2e8f0",
        block_title_text_color_dark="#e2e8f0",
        body_text_color="#cbd5e1",
        body_text_color_dark="#cbd5e1",
        body_text_color_subdued="#64748b",
        body_text_color_subdued_dark="#64748b",
        input_background_fill="#111d33",
        input_background_fill_dark="#111d33",
        input_border_color="#1e3050",
        input_border_color_dark="#1e3050",
        button_primary_background_fill="linear-gradient(135deg, #0ea5e9, #0284c7)",
        button_primary_background_fill_dark="linear-gradient(135deg, #0ea5e9, #0284c7)",
        button_primary_text_color="white",
        button_primary_text_color_dark="white",
        button_secondary_background_fill="rgba(14,165,233,0.08)",
        button_secondary_background_fill_dark="rgba(14,165,233,0.08)",
        button_secondary_text_color="#7dd3fc",
        button_secondary_text_color_dark="#7dd3fc",
        button_secondary_border_color="rgba(14,165,233,0.2)",
        button_secondary_border_color_dark="rgba(14,165,233,0.2)",
        border_color_accent="#0ea5e9",
        border_color_accent_dark="#0ea5e9",
        color_accent_soft="rgba(14,165,233,0.1)",
        color_accent_soft_dark="rgba(14,165,233,0.1)",
        shadow_spread="0px",
        checkbox_label_text_color="#94a3b8",
        checkbox_label_text_color_dark="#94a3b8",
    ),
    css=CUSTOM_CSS,
) as demo:

    # ═══════════ STATE ═══════════════════════════════════════════════════════
    findings_state = gr.State("")
    context_state = gr.State("")
    img_path_state = gr.State("")
    soap_state = gr.State("")
    loc_state = gr.State("")

    # ═══════════ HEADER ══════════════════════════════════════════════════════
    gr.HTML("""
    <div class="header-banner">
        <h1>🫁 Gemma <span class="accent">Co-Radiologist</span>
            <span class="tag">Radiology DeepResearch Agent</span>
        </h1>
        <p>Advancing the Diagnostic Medicine Frontier · Declarative DataFlow Agent · MedGemma 4B · MedSigLIP-448 · nanoathens SDK</p>
    </div>
    """)

    # ═══════════ TOP CONTROLS ════════════════════════════════════════════════
    with gr.Row():
        patient_id = gr.Dropdown(choices=_patient_ids, value="RAD001", label="🏥 Patient", scale=1)
        image_type = gr.Dropdown(choices=["xray", "ct", "mri"], value="xray", label="📡 Modality", scale=1)
        body_part = gr.Dropdown(choices=["chest", "head", "abdomen", "brain", "other"], value="chest", label="🦴 Body Region", scale=1)

    # ═══════════ MAIN LAYOUT ═════════════════════════════════════════════════
    with gr.Row():

        # ──── LEFT PANEL ─────────────────────────────────────────────────────
        with gr.Column(scale=4):

            image_input = gr.Image(label="Upload Medical Image", type="filepath", height=320)

            analyze_btn = gr.Button(
                "🔬  Run Deep Analysis",
                variant="primary", size="lg",
                elem_classes=["deep-analysis-btn"],
            )

            status_box = gr.HTML(
                value=_status_html("Ready", "Upload an image and click Run Deep Analysis", "info"),
            )

            with gr.Accordion("📋 Patient Records (EHR)", open=False):
                patient_info_md = gr.Markdown("Select a patient.", elem_classes=["ehr-panel"])

        # ──── RIGHT PANEL ────────────────────────────────────────────────────
        with gr.Column(scale=6):

            with gr.Tabs():

                with gr.TabItem("📝 Clinical Narrative"):
                    narrative_output = gr.Markdown(
                        "Run Deep Analysis to generate the clinical narrative.",
                        elem_classes=["narrative-box"],
                    )

                with gr.TabItem("🎯 Localization"):
                    localized_image = gr.Image(label="Annotated Image", height=380, interactive=False)
                    localization_info = gr.Markdown("Bounding boxes will appear after analysis.")

                with gr.TabItem("📋 SOAP Report"):
                    soap_output = gr.Textbox(label="SOAP Note", interactive=False, lines=18, elem_classes=["scroll-output"])

                with gr.TabItem("📊 Longitudinal"):
                    gr.Markdown("**Side-by-Side: Prior vs Current Study**")
                    with gr.Row():
                        prior_image_display = gr.Image(label="📁 Prior Study", height=250, interactive=False)
                        current_image_display = gr.Image(label="📷 Current Study", height=250, interactive=False)
                    longitudinal_output = gr.Markdown("Interval change assessment will appear here.")

                with gr.TabItem("📄 Export PDF"):
                    pdf_status = gr.Markdown("PDF is generated automatically during deep analysis.")
                    pdf_download = gr.File(label="Download Report", visible=True)

                with gr.TabItem("🔗 Pipeline DAG"):
                    gr.Markdown(f"### Dataflow Graph\n{_graph_summary}")
                    gr.Dataframe(value=_graph_rows, headers=["Tool", "Type", "Inputs", "Outputs"], label="Registered Tools", interactive=False)
                    gr.Markdown(f"### Resolution Paths\n{_dag_paths}")

            # ── Similar Cases Gallery (under tabs on right side) ─────────
            with gr.Accordion("🔍 Similar Cases Retrieved", open=False):
                similar_gallery = gr.Gallery(
                    label="Top-K Similar (MedSigLIP + FAISS)",
                    columns=5, height=200, object_fit="contain",
                )

    # ═══════════ FULL-WIDTH CHAT ═════════════════════════════════════════════
    gr.HTML('<div class="chat-section-divider"></div>')
    gr.Markdown("### 💬 Clinician Chat")

    chat_mode = gr.Radio(
        choices=["Follow-up", "Revise"],
        value="Follow-up",
        label="Mode",
        info="Follow-up: ask clarifying questions · Revise: request changes (re-generates narrative, localization & PDF)",
    )

    chatbot = gr.Chatbot(label="Conversation", height=260, type="messages")

    with gr.Row():
        chat_input = gr.Textbox(
            placeholder="Ask about findings, request clarification, or provide revision feedback...",
            label="Message", scale=5, lines=1,
        )
        chat_send_btn = gr.Button("Send", variant="primary", scale=1)

    # ══════════════════════════════════════════════════════════════════════════
    #  EVENT WIRING
    # ══════════════════════════════════════════════════════════════════════════

    patient_id.change(fn=get_patient_info, inputs=patient_id, outputs=patient_info_md)

    # Output list ORDER must match yield tuple positions exactly
    _analysis_outputs = [
        status_box,             # 0
        narrative_output,       # 1
        localized_image,        # 2
        localization_info,      # 3
        soap_output,            # 4
        prior_image_display,    # 5
        current_image_display,  # 6
        longitudinal_output,    # 7
        pdf_download,           # 8
        pdf_status,             # 9
        similar_gallery,        # 10
        findings_state,         # 11
        context_state,          # 12
        img_path_state,         # 13
        soap_state,             # 14
        loc_state,              # 15
    ]

    analyze_btn.click(
        fn=run_deep_analysis,
        inputs=[image_input, image_type, body_part, patient_id],
        outputs=_analysis_outputs,
    )

    _chat_outputs = [
        chatbot, chat_input, context_state,
        narrative_output, localized_image, localization_info,
        soap_output, pdf_download, pdf_status, findings_state,
    ]

    chat_send_btn.click(
        fn=chat_handler,
        inputs=[chat_input, chatbot, context_state, chat_mode,
                findings_state, img_path_state, body_part, patient_id, soap_state],
        outputs=_chat_outputs,
    )
    chat_input.submit(
        fn=chat_handler,
        inputs=[chat_input, chatbot, context_state, chat_mode,
                findings_state, img_path_state, body_part, patient_id, soap_state],
        outputs=_chat_outputs,
    )


print("✓ Gradio UI built — Gemma Co-Radiologist · Radiology DeepResearch Agent")

/tmp/ipykernel_55/1131116203.py:824: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_55/1131116203.py:824: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_55/1131116203.py:968: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversation", height=260, type="messages")


✓ Gradio UI built — Gemma Co-Radiologist · Radiology DeepResearch Agent


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  LAUNCH — REPLACES CELL 36
# ══════════════════════════════════════════════════════════════════════════════
demo.close()  # kill any previous instance

import asyncio
asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())

demo.launch(
    share=True,
    debug=True,
    show_error=True,
    server_name="0.0.0.0",
    allowed_paths=[
        KAGGLE_ROOT,           # CheXpert dataset (for similar images gallery + priors)
        "/kaggle/working",     # Generated files (localized images, PDFs)
        "/tmp",                # Temp files
    ],
)